# Bruna: validação externa CREMA-D → RAVDESS

**Estado distribuído: não executado.** Os resultados só existirão depois das células de download, extração e inferência. Este notebook complementa os Colabs 10 (figuras) e 11 (45 ajustes/IG), sem alterá-los.

O teste usa os checkpoints originais congelados EmoT e CNN1D treinados no CREMA-D. Não há fine-tuning, seleção de checkpoint ou normalização ajustada ao RAVDESS. O projeto já usou RAVDESS em etapas históricas; esta é uma avaliação externa entre bases definida após esse desenvolvimento, sem alegação de base inteiramente inédita ou pré-registro.

São 24 atores, vídeos somente visuais de fala e seis emoções comuns, totalizando 1.056 estímulos nominais. Calm e surprised ficam fora; calm não é fundido a neutral. As versões AV e VO do mesmo estímulo não são duplicadas.

Os ZIPs oficiais por ator somam aproximadamente **13,36 GB transferidos**. O processamento mantém um ZIP temporário por vez e persiste somente os vídeos selecionados no Drive. A extração é em CPU pelo MediaPipe Tasks/IMAGE 0.10.35 e OpenCV headless 4.13.0.92; a inferência usa GPU se disponível. Use uma sessão própria, sem interromper um treinamento do Colab 11. O tempo depende do download e da extração de 105.600 quadros amostrados, com retomada por ator e vídeo.

Recuperação do ambiente em 07/11/2026: MediaPipe 1.0.0/OpenCV 5.0 produziram erro médio acima da tolerância tanto na sessão CPU quanto na A100. O notebook original de agosto carregou o cache consolidado, criado em 25/06/2026, antes dessas versões. As versões aqui fixadas eram as mais recentes disponíveis na data do cache; são uma hipótese de recuperação do ambiente, não um registro histórico comprovado. A aceitação depende de repetir a conferência numérica com as mesmas tolerâncias, sem observar desempenho RAVDESS e sem afrouxar os critérios. Os diagnósticos anteriores foram preservados.

Dados: https://zenodo.org/records/1188976 — Livingstone e Russo (2018), DOI 10.1371/journal.pone.0196391. Observe os termos CC BY-NC-SA 4.0. Os vídeos não devem ser incluídos no repositório de notebooks.


In [ ]:
%pip install -q "mediapipe==0.10.35" "opencv-python-headless==4.13.0.92" requests


In [ ]:
from pathlib import Path
import sys, json, hashlib, platform, shutil
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

BASE_DIR=Path('/content/drive/MyDrive/cremad_fairness_xai')
OUTPUT_DIR=BASE_DIR/'revision_ravdess_external_20261107'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
RUN_ID='20260809_223452_c0613d3c'
required=[BASE_DIR/'cache/face_landmarker.task', BASE_DIR/'cache/cremad_landmarks_T100_semantic.npz',
          BASE_DIR/'runs'/RUN_ID/'outputs/tables/manifest_valid_with_splits.csv']
required += [BASE_DIR/'runs'/RUN_ID/'outputs/models'/name for name in ('best_EmoT.pt','best_TemporalCNN1D.pt')]
missing=[str(p) for p in required if not p.is_file()]
if missing:raise FileNotFoundError('Artefatos canônicos ausentes: '+str(missing))
import psutil
if psutil.virtual_memory().available<4*2**30:raise RuntimeError('Necessários ao menos 4 GiB de RAM livre.')
if shutil.disk_usage('/content').free<4*2**30:raise RuntimeError('Necessários ao menos 4 GiB de disco temporário livre.')
print('Python:',platform.python_version(),'Torch:',torch.__version__,'GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Saídas:',OUTPUT_DIR)


In [ ]:
MODULES={'bruna_revision_models.py': '"""Canonical model classes copied verbatim from executed notebook 08.\nRun: 20260809_223452_c0613d3c. No architectural changes.\n"""\nimport torch\nimport torch.nn as nn\n\nclass FlatMLP(nn.Module):\n    def __init__(self, T, D, n_classes, hidden=256, dropout=0.30):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Flatten(),\n            nn.Linear(T * D, hidden),\n            nn.ReLU(),\n            nn.Dropout(dropout),\n            nn.Linear(hidden, hidden // 2),\n            nn.ReLU(),\n            nn.Dropout(dropout),\n            nn.Linear(hidden // 2, n_classes),\n        )\n    def forward(self, x):\n        return self.net(x)\n\nclass TemporalCNN1D(nn.Module):\n    def __init__(self, T, D, n_classes, n_filters=96, dropout=0.30):\n        super().__init__()\n        self.features = nn.Sequential(\n            nn.Conv1d(D, n_filters, kernel_size=5, padding=2),\n            nn.ReLU(),\n            nn.BatchNorm1d(n_filters),\n            nn.MaxPool1d(2),\n            nn.Conv1d(n_filters, n_filters * 2, kernel_size=3, padding=1),\n            nn.ReLU(),\n            nn.BatchNorm1d(n_filters * 2),\n            nn.MaxPool1d(2),\n            nn.Conv1d(n_filters * 2, n_filters * 2, kernel_size=3, padding=1),\n            nn.ReLU(),\n            nn.AdaptiveAvgPool1d(1),\n        )\n        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(n_filters * 2, n_classes))\n    def forward(self, x):\n        x = x.permute(0, 2, 1)\n        return self.classifier(self.features(x).squeeze(-1))\n\nclass PedroStyleTransformerBlockWithAttention(nn.Module):\n    def __init__(self, d_model, n_heads, dim_ff, dropout):\n        super().__init__()\n        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)\n        self.drop1 = nn.Dropout(dropout)\n        self.norm1 = nn.LayerNorm(d_model, eps=1e-6)\n        self.ff = nn.Sequential(nn.Linear(d_model, dim_ff), nn.GELU(), nn.Linear(dim_ff, d_model))\n        self.drop2 = nn.Dropout(dropout)\n        self.norm2 = nn.LayerNorm(d_model, eps=1e-6)\n    def forward(self, x, return_attention=False):\n        attn_out, attn_w = self.attn(x, x, x, need_weights=return_attention, average_attn_weights=False)\n        x = self.norm1(x + self.drop1(attn_out))\n        x = self.norm2(x + self.drop2(self.ff(x)))\n        return (x, attn_w) if return_attention else (x, None)\n\nclass EmoTransformer(nn.Module):\n    def __init__(\n        self,\n        D,\n        n_classes,\n        d_model=192,\n        n_heads=3,\n        n_layers=6,\n        dim_ff=768,\n        dropout=0.30,\n        mlp_head_size=256,\n        max_len=100,\n    ):\n        super().__init__()\n        self.input_proj = nn.Linear(D, d_model)\n        self.cls_token = nn.Parameter(torch.empty(1, 1, d_model))\n        self.pos_embed = nn.Parameter(torch.empty(1, max_len + 1, d_model))\n        self.blocks = nn.ModuleList([PedroStyleTransformerBlockWithAttention(d_model, n_heads, dim_ff, dropout) for _ in range(n_layers)])\n        self.head = nn.Sequential(nn.Linear(d_model, mlp_head_size), nn.Linear(mlp_head_size, n_classes))\n        self.reset_parameters()\n\n    def reset_parameters(self):\n        nn.init.kaiming_normal_(self.cls_token)\n        nn.init.normal_(self.pos_embed, mean=0.0, std=0.02)\n\n    def forward(self, x, return_attention=False):\n        B = x.size(0)\n        x = self.input_proj(x)\n        cls = self.cls_token.expand(B, -1, -1)\n        x = torch.cat([cls, x], dim=1)\n        x = x + self.pos_embed[:, : x.size(1), :]\n        attentions = []\n        for block in self.blocks:\n            x, attn = block(x, return_attention=return_attention)\n            if return_attention:\n                attentions.append(attn)\n        logits = self.head(x[:, 0])\n        return (logits, attentions) if return_attention else logits\n\ndef count_parameters(model):\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)\n\n\n', 'bruna_revision_experiments.py': '"""Controlled supplementary experiments for Bruna\'s final dissertation revision.\n\nNothing in this module downloads data, modifies canonical artifacts or runs on import.\nCPU-only helpers are independently testable; training/IG import PyTorch lazily.\n"""\nfrom __future__ import annotations\n\nimport copy\nimport gc\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport random\nimport shutil\nimport sys\nimport time\nimport uuid\n\nimport numpy as np\nimport pandas as pd\n\nCANONICAL_RUN = "20260809_223452_c0613d3c"\nPROTOCOL_VERSION = "bruna-controlled-revision-20261107-v1"\nCLASSES = ["angry", "disgust", "fear", "happy", "neutral", "sad"]\nVARIANTS = ("EmoT_full", "TemporalCNN1D_full", "EmoT_position")\nHYPERPARAMETERS = {\n    "epochs": 3000, "patience": 50, "batch_size": 256, "learning_rate_base": 1.0,\n    "weight_decay": 3e-5, "label_smoothing": 0.10, "warmup_fraction": 0.10,\n    "gradient_clip_value": 0.5, "gradient_clip_norm": 1.0,\n    "d_model": 192, "n_heads": 3, "n_layers": 6, "dim_ff": 768,\n    "dropout": 0.30, "mlp_head_size": 256, "max_len": 100,\n}\n\n\ndef hash_file(path, block_size=2**20):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for block in iter(lambda: f.read(block_size), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef fingerprint(config):\n    return hashlib.sha256(json.dumps(config, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()\n\n\ndef array_digest(array):\n    a = np.ascontiguousarray(array)\n    h = hashlib.sha256(str(a.dtype).encode() + str(a.shape).encode())\n    h.update(a.tobytes())\n    return h.hexdigest()\n\n\ndef atomic_json(path, data):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + "." + uuid.uuid4().hex + ".partial")\n    try:\n        tmp.write_text(json.dumps(data, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")\n        os.replace(tmp, path)\n    finally:\n        tmp.unlink(missing_ok=True)\n\n\ndef atomic_csv(path, frame):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + "." + uuid.uuid4().hex + ".partial")\n    try:\n        frame.to_csv(tmp, index=False)\n        os.replace(tmp, path)\n    finally:\n        tmp.unlink(missing_ok=True)\n\n\ndef atomic_npz(path, **arrays):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + "." + uuid.uuid4().hex + ".partial")\n    try:\n        with tmp.open("wb") as f:\n            np.savez_compressed(f, **arrays)\n        os.replace(tmp, path)\n    finally:\n        tmp.unlink(missing_ok=True)\n\n\ndef require_compatible(existing, config):\n    """Reject any existing job belonging to a different protocol/configuration."""\n    expected = fingerprint(config)\n    if existing.get("fingerprint") != expected:\n        raise ValueError("Existing result/checkpoint has a different fingerprint; use a new output directory.")\n    return expected\n\n\ndef lock_config(directory, config):\n    directory = Path(directory); directory.mkdir(parents=True, exist_ok=True)\n    path = directory / "config.json"\n    if path.exists():\n        require_compatible(json.loads(path.read_text(encoding="utf-8")), config)\n    else:\n        atomic_json(path, {"fingerprint": fingerprint(config), "config": config})\n    return fingerprint(config)\n\n\ndef temporal_features(x, position_only=False):\n    """Exact canonical finite differences; no physical-time derivative claim."""\n    x = np.asarray(x, dtype=np.float32)\n    if x.ndim != 3 or x.shape[1] != 100 or x.shape[2] != 310:\n        raise ValueError(f"Expected [N,100,310], found {x.shape}.")\n    if not np.isfinite(x).all():\n        raise ValueError("Nonfinite landmarks.")\n    if position_only:\n        return x.copy()\n    out = np.zeros((len(x), 100, 1240), dtype=np.float32)\n    out[:, :, :310] = x\n    out[:, 1:, 310:620] = x[:, 1:] - x[:, :-1]\n    out[:, 1:-1, 620:930] = x[:, 2:] - 2*x[:, 1:-1] + x[:, :-2]\n    out[:, 2:-2, 930:] = x[:, 4:] - 2*x[:, 3:-1] + 2*x[:, 1:-3] - x[:, :-4]\n    return out\n\n\ndef validate_alignment(manifest, x, y, file_stems, expected_n=7441):\n    if len(manifest) != expected_n or x.shape != (expected_n, 100, 310):\n        raise ValueError("The full canonical 7,441-video tensor is required.")\n    required = {"ActorID", "file_stem", "y", "split", "Race"}\n    if required.difference(manifest.columns):\n        raise ValueError(f"Manifest missing {sorted(required.difference(manifest.columns))}")\n    if not np.array_equal(manifest["file_stem"].astype(str).to_numpy(), np.asarray(file_stems).astype(str)):\n        raise ValueError("Manifest/cache file_stems are not aligned.")\n    if not np.array_equal(manifest["y"].to_numpy(dtype=int), np.asarray(y, dtype=int)):\n        raise ValueError("Manifest/cache labels are not aligned.")\n    if manifest["ActorID"].isna().any() or manifest["file_stem"].duplicated().any():\n        raise ValueError("Missing actor or duplicate video.")\n    if expected_n == 7441 and manifest.ActorID.nunique() != 91:\n        raise ValueError("Expected the 91 canonical actors.")\n    for col in ("Race","Sex","Ethnicity","Age","split"):\n        if col in manifest and (manifest.groupby("ActorID")[col].nunique(dropna=False)>1).any():\n            raise ValueError(f"Inconsistent {col} within an actor.")\n    if np.any((np.asarray(y) < 0) | (np.asarray(y) > 5)):\n        raise ValueError("Invalid six-class labels.")\n\n\ndef validate_fold(manifest, train_idx, val_idx, test_idx):\n    """All videos and actors must occur in exactly one split of this fold."""\n    arrays = [np.asarray(z) for z in (train_idx, val_idx, test_idx)]\n    for z in arrays:\n        if z.ndim != 1 or not np.issubdtype(z.dtype, np.integer) or not len(z):\n            raise ValueError("Split indices must be nonempty integer vectors.")\n        if len(np.unique(z)) != len(z) or z.min() < 0 or z.max() >= len(manifest):\n            raise ValueError("Duplicate or invalid index.")\n    merged = np.concatenate(arrays)\n    if len(merged) != len(manifest) or not np.array_equal(np.sort(merged), np.arange(len(manifest))):\n        raise ValueError("Split indices do not partition the entire manifest.")\n    actors = [set(manifest.iloc[z]["ActorID"].tolist()) for z in arrays]\n    if any(actors[i].intersection(actors[j]) for i in range(3) for j in range(i+1, 3)):\n        raise ValueError("Actor leakage between training, validation or test.")\n    return {"train_actors": len(actors[0]), "validation_actors": len(actors[1]), "test_actors": len(actors[2])}\n\n\ndef metrics(y_true, y_pred):\n    y_true, y_pred = np.asarray(y_true, dtype=int), np.asarray(y_pred, dtype=int)\n    if y_true.shape != y_pred.shape or y_true.ndim != 1 or not len(y_true):\n        raise ValueError("Incompatible metric arrays.")\n    if min(y_true.min(), y_pred.min()) < 0 or max(y_true.max(), y_pred.max()) > 5:\n        raise ValueError("Invalid class index.")\n    cm = np.bincount(6*y_true + y_pred, minlength=36).reshape(6, 6)\n    true, predicted, tp = cm.sum(1), cm.sum(0), cm.diagonal()\n    f1 = np.divide(2*tp, true+predicted, out=np.zeros(6), where=(true+predicted)>0)\n    recall = np.divide(tp, true, out=np.zeros(6), where=true>0)\n    return {"accuracy": float(tp.sum()/len(y_true)), "macro_f1": float(f1.mean()),\n            "balanced_accuracy": float(recall[true>0].mean()), "weighted_f1": float((f1*true).sum()/len(y_true))}\n\n\ndef select_sanity_samples(manifest, n_samples=64, seed=42):\n    """Fixed holdout only; round-robin race then actor, sampling without replacement."""\n    if n_samples < 1:\n        raise ValueError("n_samples must be positive.")\n    candidates = manifest.loc[manifest["split"].eq("test")].copy()\n    if not len(candidates):\n        raise ValueError("No fixed-test samples.")\n    rng = np.random.default_rng(seed)\n    queues = {}\n    for race, race_df in candidates.groupby("Race", dropna=False, sort=True):\n        actor_ids = race_df["ActorID"].drop_duplicates().to_numpy(copy=True)\n        rng.shuffle(actor_ids)\n        queues[str(race)] = [list(rng.permutation(race_df.index[race_df["ActorID"].eq(a)].to_numpy()).astype(int)) for a in actor_ids]\n    selected, cursors = [], {race: 0 for race in queues}\n    while len(selected) < min(n_samples, len(candidates)):\n        progress = False\n        for race, actor_queues in queues.items():\n            available = [j for j, q in enumerate(actor_queues) if q]\n            if not available:\n                continue\n            start = cursors[race]\n            chosen = next((j for j in range(start, len(actor_queues)) if actor_queues[j]), available[0])\n            selected.append(actor_queues[chosen].pop())\n            cursors[race] = (chosen+1) % len(actor_queues)\n            progress = True\n            if len(selected) == min(n_samples, len(candidates)):\n                break\n        if not progress:\n            raise RuntimeError("Could not complete sanity sample selection.")\n    return np.asarray(selected, dtype=np.int64)\n\n\ndef _dependencies():\n    import torch\n    from bruna_revision_models import EmoTransformer, TemporalCNN1D\n    return torch, EmoTransformer, TemporalCNN1D\n\n\ndef _seed(seed, torch):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef _torch_load(path, torch, device="cpu"):\n    # Canonical and revision checkpoints are trusted user-owned experiment files.\n    try:\n        return torch.load(path, map_location=device, weights_only=False)\n    except TypeError:\n        return torch.load(path, map_location=device)\n\n\ndef _atomic_torch(path, payload, torch):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name+"."+uuid.uuid4().hex+".partial")\n    try:\n        torch.save(payload, tmp); os.replace(tmp, path)\n    finally:\n        tmp.unlink(missing_ok=True)\n\n\ndef _environment(torch):\n    return {"python": sys.version, "platform": platform.platform(), "numpy": np.__version__,\n            "pandas": pd.__version__, "torch": torch.__version__, "cuda": torch.version.cuda,\n            "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",\n            "cudnn": torch.backends.cudnn.version(), "cudnn_deterministic": torch.backends.cudnn.deterministic,\n            "cudnn_benchmark": torch.backends.cudnn.benchmark,\n            "deterministic_algorithms": torch.are_deterministic_algorithms_enabled()}\n\n\ndef _runtime_signature(torch):\n    env = _environment(torch)\n    return {k: env[k] for k in ("python", "numpy", "pandas", "torch", "cuda", "device", "cudnn", "cudnn_deterministic", "cudnn_benchmark", "deterministic_algorithms")}\n\n\ndef _load_sources(base_dir):\n    base = Path(base_dir)\n    run = base/"runs"/CANONICAL_RUN/"outputs"\n    cache = base/"cache/cremad_landmarks_T100_semantic.npz"\n    manifest_path = run/"tables/manifest_valid_with_splits.csv"\n    with np.load(cache, allow_pickle=False) as z:\n        x = z["X_base"].astype(np.float32)\n        y = z["y"].astype(np.int64)\n        stems = z["file_stems"].astype(str)\n        landmarks = z["selected_landmark_ids"].astype(int)\n    manifest = pd.read_csv(manifest_path)\n    validate_alignment(manifest, x, y, stems)\n    if len(landmarks) != 155 or len(np.unique(landmarks)) != 155:\n        raise ValueError("Expected 155 unique canonical landmarks.")\n    print("Hashing canonical inputs once for this execution; no source files are modified.", flush=True)\n    identity = {"cache_sha256": hash_file(cache), "manifest_sha256": hash_file(manifest_path),\n                "model_source_sha256": hash_file(Path(__file__).with_name("bruna_revision_models.py")),\n                "experiment_source_sha256": hash_file(__file__)}\n    return run, manifest, x, y, stems, landmarks, identity\n\n\ndef _model(variant, dimensions, EmoTransformer, TemporalCNN1D):\n    if variant == "TemporalCNN1D_full":\n        return TemporalCNN1D(T=100, D=dimensions, n_classes=6)\n    if variant not in VARIANTS:\n        raise ValueError(f"Unknown variant {variant}.")\n    return EmoTransformer(D=dimensions, n_classes=6)\n\n\ndef _predict(model, x, torch, device, batch_size=256):\n    model.eval(); logits = []\n    with torch.no_grad():\n        for start in range(0, len(x), batch_size):\n            xb = torch.from_numpy(x[start:start+batch_size]).to(device)\n            logits.append(model(xb).detach().cpu().numpy())\n    logits = np.concatenate(logits).astype(np.float32)\n    if not np.isfinite(logits).all():\n        raise RuntimeError("Nonfinite predictions.")\n    exp = np.exp(logits-logits.max(1, keepdims=True)); prob = exp/exp.sum(1, keepdims=True)\n    return prob.argmax(1), prob.astype(np.float32), logits\n\n\ndef _rng_state(torch):\n    return {"python_random_state": random.getstate(), "numpy_random_state": np.random.get_state(),\n            "torch_random_state": torch.get_rng_state(),\n            "cuda_random_state_all": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}\n\n\ndef _restore_rng(payload, torch):\n    random.setstate(payload["python_random_state"]); np.random.set_state(payload["numpy_random_state"])\n    torch.set_rng_state(payload["torch_random_state"].cpu())\n    if torch.cuda.is_available() and payload.get("cuda_random_state_all") is not None:\n        torch.cuda.set_rng_state_all([s.cpu() for s in payload["cuda_random_state_all"]])\n\n\ndef _fit_job(directory, config, x_train, y_train, x_val, y_val, mu, sigma, torch, EmoTransformer, TemporalCNN1D):\n    directory = Path(directory); fp = lock_config(directory, config)\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    _seed(config["seed"], torch)\n    model = _model(config["variant"], config["dimensions"], EmoTransformer, TemporalCNN1D).to(device)\n    counts = np.bincount(y_train, minlength=6).astype(np.float32)\n    weights = counts.sum()/np.maximum(counts, 1); weights /= weights.mean()\n    criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor(weights, device=device), label_smoothing=.10)\n    dataset = torch.utils.data.TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train))\n    loader = torch.utils.data.DataLoader(dataset, batch_size=256, shuffle=True, num_workers=0, pin_memory=device=="cuda")\n    optimizer = torch.optim.AdamW(model.parameters(), lr=1.0, weight_decay=3e-5)\n    warmup = max(1, int(.10*len(loader)*3000))\n    def schedule(step):\n        step = max(1, step)\n        return 192**-.5 * min(step**-.5, step*warmup**-1.5)\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)\n    scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None\n    latest, best = directory/"latest.pt", directory/"best.pt"\n    start_epoch, best_f1, best_epoch, patience, history = 1, -1.0, -1, 50, []\n    if latest.exists():\n        state = _torch_load(latest, torch)\n        require_compatible(state, config)\n        model.load_state_dict(state["model_state_dict"], strict=True)\n        optimizer.load_state_dict(state["optimizer_state_dict"])\n        scheduler.load_state_dict(state["scheduler_state_dict"])\n        if scaler is not None and state.get("scaler_state_dict") is not None:\n            scaler.load_state_dict(state["scaler_state_dict"])\n        start_epoch, best_f1 = state["epoch"]+1, state["best_f1"]\n        best_epoch, patience, history = state["best_epoch"], state["patience_left"], state["history"]\n        _restore_rng(state, torch)\n        if state["completed"]:\n            if not best.exists():\n                raise RuntimeError("Completed checkpoint has no best checkpoint.")\n            chosen = _torch_load(best, torch); require_compatible(chosen, config)\n            model.load_state_dict(chosen["model_state_dict"], strict=True)\n            return model, history, best_epoch, best_f1\n    for epoch in range(start_epoch, 3001):\n        model.train(); loss_sum, seen = 0.0, 0\n        for xb, yb in loader:\n            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            if scaler is not None:\n                with torch.amp.autocast("cuda"):\n                    loss = criterion(model(xb), yb)\n                scaler.scale(loss).backward(); scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_value_(model.parameters(), .5)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                scaler.step(optimizer); scaler.update()\n            else:\n                loss = criterion(model(xb), yb); loss.backward()\n                torch.nn.utils.clip_grad_value_(model.parameters(), .5)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()\n            scheduler.step(); loss_sum += float(loss.item())*len(yb); seen += len(yb)\n        pred, _, _ = _predict(model, x_val, torch, device)\n        val = metrics(y_val, pred)\n        row = {"epoch": epoch, "train_loss": loss_sum/seen, "learning_rate": optimizer.param_groups[0]["lr"], **{f"validation_{k}": v for k,v in val.items()}}\n        history.append(row)\n        if val["macro_f1"] > best_f1:\n            best_f1, best_epoch, patience = val["macro_f1"], epoch, 50\n            _atomic_torch(best, {"fingerprint": fp, "model_state_dict": {k:v.detach().cpu() for k,v in model.state_dict().items()},\n                               "mu": mu, "sigma": sigma, "class_names": CLASSES, "best_epoch": best_epoch, "best_val_macro_f1": best_f1, "config": config}, torch)\n        else:\n            patience -= 1\n        complete = patience <= 0 or epoch == 3000\n        payload = {"fingerprint": fp, "epoch": epoch, "best_f1": best_f1, "best_epoch": best_epoch, "patience_left": patience,\n                   "completed": complete, "history": history, "model_state_dict": {k:v.detach().cpu() for k,v in model.state_dict().items()},\n                   "optimizer_state_dict": optimizer.state_dict(), "scheduler_state_dict": scheduler.state_dict(),\n                   "scaler_state_dict": scaler.state_dict() if scaler is not None else None, **_rng_state(torch)}\n        _atomic_torch(latest, payload, torch)\n        atomic_csv(directory/"history.csv", pd.DataFrame(history))\n        if epoch == 1 or epoch % 10 == 0 or complete:\n            print(f"{config[\'variant\']} seed={config[\'seed\']} fold={config[\'fold\']} epoch={epoch} valF1={val[\'macro_f1\']:.5f} patience={patience}", flush=True)\n        if complete:\n            break\n    if not best.exists():\n        raise RuntimeError("No best checkpoint was produced.")\n    chosen = _torch_load(best, torch); require_compatible(chosen, config)\n    model.load_state_dict(chosen["model_state_dict"], strict=True)\n    return model, history, best_epoch, best_f1\n\n\ndef run_revision_training(base_dir, output_dir=None, seeds=(42, 2026, 2027), folds=(1,2,3,4,5)):\n    """Run all 45 controlled fits; retain historical fits only as separate evidence."""\n    torch, EmoTransformer, TemporalCNN1D = _dependencies()\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA is required for the definitive 45-fit revision; select a Colab GPU.")\n    seeds, folds = tuple(map(int,seeds)), tuple(map(int,folds))\n    if not seeds or not folds or len(set(seeds)) != len(seeds) or len(set(folds)) != len(folds) or not set(folds).issubset(range(1,6)):\n        raise ValueError("Seeds and fold IDs must be unique; folds must be between 1 and 5.")\n    root = Path(output_dir or Path(base_dir)/"revision_final_20261107")/"training"\n    run, manifest, x_base, y, stems, landmarks, identity = _load_sources(base_dir)\n    protocol = {"version": PROTOCOL_VERSION, "canonical_run": CANONICAL_RUN, "seeds": list(seeds), "folds": list(folds),\n                "variants": list(VARIANTS), "hyperparameters": HYPERPARAMETERS, "source": identity,\n                "runtime": _runtime_signature(torch), "historical_fits_reused": False,\n                "seed_policy": "reset RNG independently before each seed/fold/variant fit"}\n    lock_config(root, protocol); atomic_json(root/"environment.json", _environment(torch))\n    historical = root/"historical_reference"; historical.mkdir(parents=True, exist_ok=True)\n    for name in ("cv_fold_metrics.csv", "cv_summary_mean_std.csv"):\n        source, dest = run/"tables"/name, historical/name\n        if dest.exists() and hash_file(dest) != hash_file(source):\n            raise ValueError("Historical reference changed; do not overwrite it.")\n        if not dest.exists():shutil.copyfile(source, dest)\n    split_specs, test_coverage = {}, np.zeros(len(manifest), dtype=int)\n    for fold in folds:\n        split_path = run/"cross_validation"/f"fold_{fold:02d}"/"normalization_and_indices.npz"\n        with np.load(split_path, allow_pickle=False) as z:\n            tr, va, te = [z[k].astype(np.int64) for k in ("train_idx","val_idx","test_idx")]\n            saved_mu, saved_sigma = z["mu"].astype(np.float32).reshape(-1), z["sigma"].astype(np.float32).reshape(-1)\n        validate_fold(manifest, tr, va, te); test_coverage[te] += 1\n        split_specs[fold] = (tr, va, te, saved_mu, saved_sigma, hash_file(split_path))\n    if set(folds) == set(range(1,6)) and not np.all(test_coverage == 1):\n        raise ValueError("The five external tests do not cover every video exactly once.")\n    rows, prediction_frames = [], []\n    for fold in folds:\n        tr, va, te, saved_mu, saved_sigma, split_sha = split_specs[fold]\n        for position_only in (False, True):\n            dimensions = 310 if position_only else 1240\n            x_train = temporal_features(x_base[tr], position_only)\n            mu = x_train.reshape(-1, dimensions).mean(0).astype(np.float32)\n            sigma = x_train.reshape(-1, dimensions).std(0).astype(np.float32); sigma[sigma==0] = 1\n            expected_mu = saved_mu[:dimensions] if position_only else saved_mu\n            expected_sigma = saved_sigma[:dimensions] if position_only else saved_sigma\n            if not (np.allclose(mu, expected_mu, rtol=1e-5, atol=1e-6) and np.allclose(sigma, expected_sigma, rtol=1e-5, atol=1e-6)):\n                raise ValueError(f"Recomputed training-only normalization differs from canonical fold {fold}.")\n            x_train -= mu; x_train /= sigma\n            x_val, x_test = temporal_features(x_base[va], position_only), temporal_features(x_base[te], position_only)\n            x_val -= mu; x_val /= sigma; x_test -= mu; x_test /= sigma\n            variants = ("EmoT_position",) if position_only else ("EmoT_full", "TemporalCNN1D_full")\n            for seed in seeds:\n                for variant in variants:\n                    directory = root/f"seed_{seed}"/f"fold_{fold:02d}"/variant\n                    config = {"version": PROTOCOL_VERSION, "variant": variant, "seed": seed, "fold": fold, "dimensions": dimensions,\n                              "source": identity, "split_sha256": split_sha, "hyperparameters": HYPERPARAMETERS, "runtime": protocol["runtime"],\n                              "normalization_sha256": fingerprint({"mu": array_digest(mu), "sigma": array_digest(sigma)})}\n                    fp = lock_config(directory, config); result_path = directory/"result.json"\n                    if result_path.exists():\n                        result = json.loads(result_path.read_text(encoding="utf-8")); require_compatible(result, config)\n                        for name,digest in result["artifact_sha256"].items():\n                            if hash_file(directory/name) != digest:\n                                raise RuntimeError(f"Completed artifact failed hash validation: {directory/name}")\n                        with np.load(directory/"predictions.npz", allow_pickle=False) as z:\n                            if not np.array_equal(z["test_idx"], te) or not np.array_equal(z["y_true"], y[te]):\n                                raise RuntimeError("Completed prediction/manifest mismatch.")\n                        frame = pd.read_csv(directory/"predictions.csv"); row = result["metrics"]\n                    else:\n                        started = time.time()\n                        model, history, best_epoch, best_f1 = _fit_job(directory, config, x_train, y[tr], x_val, y[va], mu, sigma, torch, EmoTransformer, TemporalCNN1D)\n                        pred, prob, logits = _predict(model, x_test, torch, "cuda")\n                        frame = pd.DataFrame({"sample_idx": te,"ActorID": manifest.iloc[te]["ActorID"].to_numpy(),"fold": fold,"seed": seed,"variant": variant,"y_true": y[te],"y_pred": pred})\n                        for c,name in enumerate(CLASSES):\n                            frame[f"prob_{name}"] = prob[:,c]\n                        atomic_npz(directory/"predictions.npz", test_idx=te,y_true=y[te],y_pred=pred,y_prob=prob,logits=logits)\n                        atomic_csv(directory/"predictions.csv", frame)\n                        row = {"fold": fold,"seed": seed,"variant": variant,"dimensions": dimensions,"parameters": sum(p.numel() for p in model.parameters()),\n                               "best_epoch": best_epoch,"best_validation_macro_f1": best_f1,"elapsed_seconds": time.time()-started,**metrics(y[te],pred)}\n                        hashes = {name:hash_file(directory/name) for name in ("best.pt","latest.pt","predictions.npz","predictions.csv","history.csv")}\n                        atomic_json(result_path,{"fingerprint":fp,"status":"completed","metrics":row,"artifact_sha256":hashes})\n                        del model; gc.collect(); torch.cuda.empty_cache()\n                    rows.append(row); prediction_frames.append(frame)\n                    atomic_csv(root/"fold_metrics_progress.csv", pd.DataFrame(rows))\n            del x_train,x_val,x_test; gc.collect(); torch.cuda.empty_cache()\n    results = pd.DataFrame(rows)\n    expected = len(seeds)*len(folds)*3\n    if len(results) != expected or results.duplicated(["seed","fold","variant"]).any():\n        raise RuntimeError("Supplementary training grid is incomplete or duplicated.")\n    predictions = pd.concat(prediction_frames, ignore_index=True)\n    atomic_csv(root/"fold_metrics.csv",results); atomic_csv(root/"pooled_predictions.csv",predictions)\n    seed_summaries = []\n    for (seed,variant), group in predictions.groupby(["seed","variant"]):\n        if set(folds) == set(range(1,6)) and (len(group)!=len(manifest) or group.sample_idx.nunique()!=len(manifest)):\n            raise RuntimeError("Supplementary OOF coverage is incomplete.")\n        seed_summaries.append({"seed":int(seed),"variant":variant,"n_videos":len(group),**metrics(group.y_true,group.y_pred)})\n    seed_metrics = pd.DataFrame(seed_summaries)\n    atomic_csv(root/"pooled_metrics_by_seed.csv",seed_metrics)\n    by_seed = results.groupby(["seed","variant"])[[\'macro_f1\',\'accuracy\']].agg([\'mean\',\'std\']).reset_index()\n    by_seed.columns = [\'_\'.join(c).rstrip(\'_\') if isinstance(c,tuple) else c for c in by_seed.columns]\n    atomic_csv(root/"fold_summary_by_seed.csv",by_seed)\n    across_seeds = seed_metrics.groupby("variant")[[\'macro_f1\',\'accuracy\']].agg([\'mean\',\'std\']).reset_index()\n    across_seeds.columns = [\'_\'.join(c).rstrip(\'_\') if isinstance(c,tuple) else c for c in across_seeds.columns]\n    atomic_csv(root/"seed_summary_pooled_metrics.csv",across_seeds)\n    summary = results.groupby("variant")[[\'macro_f1\',\'accuracy\']].agg([\'mean\',\'std\']).reset_index()\n    summary.columns = [\'_\'.join(c).rstrip(\'_\') if isinstance(c,tuple) else c for c in summary.columns]\n    atomic_csv(root/"summary_descriptive.csv",summary)\n    paired = results.pivot(index=["seed","fold"],columns="variant",values="macro_f1").reset_index()\n    paired["delta_EmoT_minus_CNN"] = paired.EmoT_full-paired.TemporalCNN1D_full\n    paired["delta_full_minus_position"] = paired.EmoT_full-paired.EmoT_position\n    atomic_csv(root/"paired_fold_seed_differences.csv",paired)\n    final_grid = seeds==(42,2026,2027) and set(folds)==set(range(1,6))\n    completion = {"status":"completed" if final_grid else "partial_grid_only", "fits":len(results),"definitive_grid_complete":final_grid,\n                  "protocol_fingerprint":fingerprint(protocol),"historical_fits_reused":False,\n                  "limitations":["Fold training sets overlap; 15 seed/fold metrics are not independent observations.","Three seeds describe limited training variability; no significance claim is generated.","Architecture parameter count changes when input dimensionality changes.","CUDA operators can retain nondeterminism; environment flags are recorded."]}\n    atomic_json(root/"completion.json",completion)\n    return completion\n\n\ndef _integrated_gradients(model, x, target, torch, device, steps=16, batch_size=4):\n    """Canonical right-Riemann 16-step IG on logits, preserving signed values."""\n    model.eval(); output=[]\n    alphas=torch.linspace(0.0,1.0,steps+1,device=device)[1:]\n    for start in range(0,len(x),batch_size):\n        xb=torch.tensor(x[start:start+batch_size],dtype=torch.float32,device=device)\n        labels=torch.tensor(target[start:start+batch_size],dtype=torch.long,device=device)\n        total=torch.zeros_like(xb)\n        for alpha in alphas:\n            point=(alpha*xb).detach().requires_grad_(True)\n            logits=model(point); selected=logits[torch.arange(len(point),device=device),labels].sum()\n            grad=torch.autograd.grad(selected,point,retain_graph=False,create_graph=False)[0]\n            total += grad.detach()\n        output.append((xb*total/steps).detach().cpu().numpy())\n    return np.concatenate(output)\n\n\ndef _average_ranks(values):\n    values=np.asarray(values); order=np.argsort(values,kind="mergesort"); sorted_values=values[order]\n    starts=np.r_[0,np.flatnonzero(sorted_values[1:]!=sorted_values[:-1])+1]\n    ends=np.r_[starts[1:],len(values)]; ranks=np.empty(len(values),dtype=float)\n    ranks[order]=np.repeat((starts+ends-1)/2,ends-starts)\n    return ranks\n\n\ndef compare_maps(original, changed, top_k):\n    a,b=np.asarray(original,dtype=float).reshape(-1),np.asarray(changed,dtype=float).reshape(-1)\n    if a.shape!=b.shape or not len(a) or not np.isfinite(a).all() or not np.isfinite(b).all():\n        raise ValueError("Invalid attribution maps.")\n    norms=np.linalg.norm(a)*np.linalg.norm(b)\n    cosine=float(np.dot(a,b)/norms) if norms else None\n    ra,rb=_average_ranks(a),_average_ranks(b);ra-=ra.mean();rb-=rb.mean()\n    rnorm=np.linalg.norm(ra)*np.linalg.norm(rb)\n    spearman=float(np.dot(ra,rb)/rnorm) if rnorm else None\n    k=min(int(top_k),len(a))\n    if k<1:raise ValueError("top_k must be positive.")\n    # Stable tie-breaking is deterministic and explicitly recorded.\n    ia=set(np.argsort(-a,kind="stable")[:k]);ib=set(np.argsort(-b,kind="stable")[:k])\n    return {"cosine":cosine,"spearman":spearman,"top_k":k,"jaccard_top_k":len(ia&ib)/len(ia|ib),\n            "original_nonzero":int(np.count_nonzero(a)),"randomized_nonzero":int(np.count_nonzero(b))}\n\n\ndef _attribution_views(attrs):\n    # Canonical plots aggregate absolute magnitudes, so primary sanity comparisons do too.\n    maps=np.abs(attrs).reshape(len(attrs),100,4,155,2).sum(axis=(2,4))\n    return {"landmark_time_absolute":maps,"landmark_absolute":maps.sum(1),"temporal_absolute":maps.sum(2)}\n\n\ndef run_parameter_sanity(base_dir, output_dir=None, n_samples=64, random_seeds=(11,29,47)):\n    """Cascading parameter randomization; fixed original targets and holdout inputs."""\n    random_seeds=tuple(map(int,random_seeds))\n    if not random_seeds or len(set(random_seeds))!=len(random_seeds):\n        raise ValueError("Randomization seeds must be nonempty and unique.")\n    torch,EmoTransformer,_=_dependencies(); device="cuda" if torch.cuda.is_available() else "cpu"\n    run,manifest,x_base,y,stems,landmarks,identity=_load_sources(base_dir)\n    original_path=run/"models/best_EmoT.pt"; payload=_torch_load(original_path,torch)\n    if list(payload["class_names"])!=CLASSES:raise ValueError("Checkpoint class order mismatch.")\n    mu,sigma=[np.asarray(payload[k],dtype=np.float32).reshape(-1) for k in ("mu","sigma")]\n    if mu.shape!=(1240,) or sigma.shape!=(1240,) or not np.isfinite(mu).all() or not np.isfinite(sigma).all() or (sigma<=0).any():\n        raise ValueError("Invalid canonical checkpoint normalization.")\n    sample_idx=select_sanity_samples(manifest,n_samples,seed=42)\n    x=temporal_features(x_base[sample_idx]);x-=mu;x/=sigma\n    del x_base;gc.collect()\n    model=EmoTransformer(D=1240,n_classes=6).to(device);model.load_state_dict(payload["model_state_dict"],strict=True)\n    target,prob,logits=_predict(model,x,torch,device)\n    stages=[("head",("head.",))]+[(f"head_blocks_{layer+1}_to_6",(f"blocks.{layer}.",)) for layer in range(5,-1,-1)]+[("all_parameters",("input_proj.","cls_token","pos_embed"))]\n    config={"version":PROTOCOL_VERSION,"source":identity,"checkpoint_sha256":hash_file(original_path),"sample_idx":sample_idx.tolist(),\n            "target_classes":target.tolist(),"steps":16,"random_seeds":list(map(int,random_seeds)),"stages":[s[0] for s in stages],"runtime":_runtime_signature(torch),\n            "reference":"zero in standardized feature space","target":"original model predicted class, fixed for all stages",\n            "quadrature":"16 right-Riemann points alpha=1/16,...,1","primary_maps":"absolute canonical magnitudes",\n            "top_k_tie_policy":"stable original feature index","selection":"fixed test only; round robin race then actor; purposive balanced audit cohort"}\n    root=Path(output_dir or Path(base_dir)/"revision_final_20261107")/"parameter_sanity"\n    fp=lock_config(root,config);atomic_json(root/"environment.json",_environment(torch))\n    selection=manifest.iloc[sample_idx][["file_stem","ActorID","Race","label_name"]].copy()\n    selection.insert(0,"sample_idx",sample_idx);selection["original_target_class"]=target;selection["original_predicted_probability"]=prob[np.arange(len(target)),target]\n    atomic_csv(root/"sample_manifest.csv",selection)\n    baseline_path, baseline_meta = root/"baseline_attributions.npz", root/"baseline.json"\n    if baseline_meta.exists():\n        cached = json.loads(baseline_meta.read_text(encoding="utf-8"));require_compatible(cached,config)\n        if hash_file(baseline_path)!=cached["sha256"]:raise RuntimeError("Baseline attribution hash mismatch.")\n        with np.load(baseline_path,allow_pickle=False) as z:\n            if not np.array_equal(z["sample_idx"],sample_idx) or not np.array_equal(z["target_classes"],target):\n                raise ValueError("Baseline sample/target mismatch.")\n            original_views=_attribution_views(z["signed_ig"])\n    else:\n        baseline=_integrated_gradients(model,x,target,torch,device)\n        original_views=_attribution_views(baseline)\n        atomic_npz(baseline_path,sample_idx=sample_idx,target_classes=target,signed_ig=baseline,\n                   landmark_time_absolute=original_views["landmark_time_absolute"],selected_landmark_ids=landmarks)\n        atomic_json(baseline_meta,{"fingerprint":fp,"sha256":hash_file(baseline_path)})\n        del baseline;gc.collect()\n    rows=[]\n    for seed in random_seeds:\n        _seed(int(seed),torch)\n        fresh=EmoTransformer(D=1240,n_classes=6)\n        randomized_state=copy.deepcopy(payload["model_state_dict"])\n        for stage,prefixes in stages:\n            stage_path=root/f"seed_{seed}"/f"{stage}.json"\n            if stage_path.exists():\n                cached=json.loads(stage_path.read_text(encoding="utf-8"));require_compatible(cached,config)\n                if cached.get("random_seed")!=int(seed) or cached.get("stage")!=stage:\n                    raise ValueError("Cached sanity result belongs to a different stage/seed.")\n                maps_path=stage_path.with_suffix(\'.npz\')\n                if hash_file(maps_path)!=cached["maps_sha256"]:raise RuntimeError("Sanity artifact hash mismatch.")\n                rows.extend(cached["rows"])\n            # Even when a stage is reused, replay the cumulative parameter replacement.\n            fresh_state=fresh.state_dict()\n            replaced=[]\n            for name,value in fresh_state.items():\n                if any(name.startswith(prefix) for prefix in prefixes):\n                    randomized_state[name]=value.clone();replaced.append(name)\n            if not replaced:raise RuntimeError(f"No parameters matched stage {stage}.")\n            if stage_path.exists():continue\n            model.load_state_dict(randomized_state,strict=True)\n            changed=_integrated_gradients(model,x,target,torch,device);views=_attribution_views(changed)\n            new_rows=[]\n            for map_name,original in original_views.items():\n                k=1000 if map_name=="landmark_time_absolute" else 20\n                for i,sample in enumerate(sample_idx):\n                    new_rows.append({"random_seed":int(seed),"stage":stage,"sample_idx":int(sample),"ActorID":int(manifest.iloc[sample].ActorID),\n                                     "Race":str(manifest.iloc[sample].Race),"original_target_class":int(target[i]),"map":map_name,\n                                     **compare_maps(original[i],views[map_name][i],k)})\n            maps_path=stage_path.with_suffix(\'.npz\')\n            atomic_npz(maps_path,sample_idx=sample_idx,target_classes=target,**views)\n            atomic_json(stage_path,{"fingerprint":fp,"status":"completed","random_seed":int(seed),"stage":stage,"rows":new_rows,"maps_sha256":hash_file(maps_path),"parameters_newly_randomized":replaced})\n            rows.extend(new_rows);del changed,views;gc.collect()\n            if torch.cuda.is_available():torch.cuda.empty_cache()\n            print(f"IG parameter sanity: seed={seed} stage={stage} complete",flush=True)\n        del fresh,randomized_state\n    frame=pd.DataFrame(rows)\n    expected=len(random_seeds)*len(stages)*len(sample_idx)*3\n    if len(frame)!=expected or frame.duplicated(["random_seed","stage","sample_idx","map"]).any():\n        raise RuntimeError("Parameter sanity grid incomplete or duplicated.")\n    atomic_csv(root/"sample_comparisons.csv",frame)\n    summary=frame.groupby(["stage","map"])[["cosine","spearman","jaccard_top_k"]].agg(["mean","std","min","max"]).reset_index()\n    summary.columns=[\'_\'.join(c).rstrip(\'_\') if isinstance(c,tuple) else c for c in summary.columns]\n    atomic_csv(root/"summary.csv",summary)\n    is_final=int(n_samples)==64 and tuple(random_seeds)==(11,29,47) and len(sample_idx)==64\n    result={"status":"completed" if is_final else "partial_grid_only","definitive_grid_complete":is_final,"randomized_stages":len(stages),\n            "random_initializations":len(random_seeds),"n_samples":len(sample_idx),"fingerprint":fp,\n            "limitations":["Numerical implementation agreement with Captum and parameter dependence are different properties.",\n                           "No automatic pass/fail threshold was selected after observing these comparisons.",\n                           "Low map similarity is not a universal certificate of explanation validity.",\n                           "The balanced fixed-holdout audit cohort does not estimate population fairness.",\n                           "Primary maps aggregate absolute magnitudes; signed baseline IG is also persisted."]}\n    atomic_json(root/"completion.json",result)\n    return result\n', 'bruna_revision_statistics.py': '"""Actor-cluster analysis of preserved CREMA-D out-of-fold predictions.\n\nDependencies are limited to NumPy and pandas. This module never trains a model.\nThe intervals are conditional on the saved out-of-fold fitted models, not a\nreplacement for repeated training or external validation.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nCLASSES = ("angry", "disgust", "fear", "happy", "neutral", "sad")\nATTRIBUTES = ("Race", "Ethnicity", "Sex", "age_bin")\nPROB_COLUMNS = tuple("prob_" + name for name in CLASSES)\n\n\ndef metrics_from_confusion(cm: np.ndarray) -> dict[str, np.ndarray]:\n    """Metrics with six prespecified classes; undefined class F1 is zero."""\n    cm = np.asarray(cm, dtype=float)\n    if cm.shape[-2:] != (6, 6) or np.any(cm < 0) or not np.isfinite(cm).all():\n        raise ValueError("Expected finite nonnegative [...,6,6] confusion matrices")\n    tp = np.diagonal(cm, axis1=-2, axis2=-1)\n    true = cm.sum(axis=-1)\n    pred = cm.sum(axis=-2)\n    f1 = np.divide(2 * tp, true + pred, out=np.zeros_like(tp), where=(true + pred) > 0)\n    recall = np.divide(tp, true, out=np.zeros_like(tp), where=true > 0)\n    total = cm.sum(axis=(-1, -2))\n    accuracy = np.divide(tp.sum(axis=-1), total, out=np.zeros_like(total), where=total > 0)\n    return {"accuracy": accuracy, "macro_f1": f1.mean(axis=-1), "balanced_accuracy": recall.mean(axis=-1)}\n\n\ndef confusion(y_true, y_pred, weights=None):\n    y_true = np.asarray(y_true, dtype=int)\n    y_pred = np.asarray(y_pred, dtype=int)\n    if len(y_true) != len(y_pred) or np.any((y_true < 0) | (y_true > 5)) or np.any((y_pred < 0) | (y_pred > 5)):\n        raise ValueError("Invalid labels or inconsistent lengths")\n    return np.bincount(6 * y_true + y_pred, weights=weights, minlength=36).reshape(6, 6)\n\n\ndef actor_confusions(frame: pd.DataFrame, actor_ids: np.ndarray) -> np.ndarray:\n    lookup = {actor: i for i, actor in enumerate(actor_ids)}\n    codes = frame["ActorID"].map(lookup)\n    if codes.isna().any():\n        raise ValueError("Prediction refers to an unknown actor")\n    index = codes.to_numpy(int) * 36 + frame["y_true"].to_numpy(int) * 6 + frame["y_pred"].to_numpy(int)\n    return np.bincount(index, minlength=len(actor_ids) * 36).reshape(len(actor_ids), 6, 6)\n\n\ndef validate_inputs(predictions, manifest, assignments, expected_samples=None, expected_actors=None):\n    """Align sample_idx to the persisted manifest and audit identity/fold labels."""\n    required = {"sample_idx", "fold", "model", "y_true", "y_pred", *PROB_COLUMNS}\n    if not required.issubset(predictions.columns):\n        raise ValueError(f"Missing prediction columns: {sorted(required - set(predictions.columns))}")\n    if not {"ActorID", "y", "file_stem", *ATTRIBUTES}.issubset(manifest.columns):\n        raise ValueError("Manifest is missing labels, identities or demographic attributes")\n    if not {"ActorID", "outer_test_fold", *ATTRIBUTES}.issubset(assignments.columns):\n        raise ValueError("Assignments are missing identities, folds or demographics")\n    if manifest["file_stem"].duplicated().any() or assignments["ActorID"].duplicated().any():\n        raise ValueError("Manifest sample or actor assignment is duplicated")\n    if expected_samples is not None and len(manifest) != expected_samples:\n        raise ValueError("Unexpected manifest size")\n    if expected_actors is not None and manifest["ActorID"].nunique() != expected_actors:\n        raise ValueError("Unexpected actor count")\n    for attr in ATTRIBUTES:\n        if manifest.groupby("ActorID")[attr].nunique(dropna=False).gt(1).any():\n            raise ValueError(f"Demographic category varies inside actor: {attr}")\n    if set(manifest.ActorID) != set(assignments.ActorID):\n        raise ValueError("Actor sets disagree between manifest and assignments")\n    actors = manifest.groupby("ActorID", sort=True).first()[list(ATTRIBUTES)]\n    actors = actors.join(assignments.set_index("ActorID")[["outer_test_fold"]])\n    reference = assignments.set_index("ActorID")\n    for attr in ATTRIBUTES:\n        if not actors[attr].fillna("Unknown").astype(str).equals(reference.loc[actors.index, attr].fillna("Unknown").astype(str)):\n            raise ValueError(f"Actor demographic metadata disagrees: {attr}")\n    by_model = {}\n    expected = np.arange(len(manifest))\n    for model, frame in predictions.groupby("model", sort=True):\n        f = frame.sort_values("sample_idx").reset_index(drop=True)\n        if not np.array_equal(f.sample_idx.to_numpy(), expected):\n            raise ValueError(f"OOF samples not covered once for {model}")\n        if not np.array_equal(f.y_true.to_numpy(), manifest.y.to_numpy()):\n            raise ValueError(f"True labels disagree with manifest for {model}")\n        probs = f[list(PROB_COLUMNS)].to_numpy(float)\n        if not np.isfinite(probs).all() or probs.min() < -1e-7 or probs.max() > 1 + 1e-7 or not np.allclose(probs.sum(1), 1, atol=2e-6):\n            raise ValueError(f"Invalid probabilities for {model}")\n        if not np.array_equal(probs.argmax(1), f.y_pred.to_numpy()):\n            raise ValueError(f"Predicted labels disagree with probability argmax for {model}")\n        f["ActorID"] = manifest.ActorID.to_numpy()\n        for attr in ATTRIBUTES:\n            f[attr] = manifest[attr].fillna("Unknown").astype(str).to_numpy()\n        actor_folds = f.ActorID.map(actors.outer_test_fold)\n        if not np.array_equal(f.fold.to_numpy(), actor_folds.to_numpy()):\n            raise ValueError(f"Fold assignments disagree for {model}")\n        if f.groupby("ActorID").fold.nunique().gt(1).any():\n            raise ValueError(f"Actor identity leaks across OOF folds for {model}")\n        by_model[str(model)] = f\n    if not {"EmoT", "TemporalCNN1D"}.issubset(by_model):\n        raise ValueError("EmoT and TemporalCNN1D are required")\n    return by_model, actors.reset_index()\n\n\ndef cluster_bootstrap_pair(cm_a, cm_b, n_boot=5000, seed=20261107, actor_equal_weight=False):\n    """Resample whole actors and apply identical actor multiplicities to models."""\n    a, b = np.asarray(cm_a, float), np.asarray(cm_b, float)\n    if a.shape != b.shape or a.ndim != 3 or a.shape[1:] != (6, 6):\n        raise ValueError("Paired actor confusion arrays must have identical [actors,6,6] shapes")\n    if not np.array_equal(a.sum((1, 2)), b.sum((1, 2))):\n        raise ValueError("Models have different per-actor support")\n    if np.any(a.sum((1, 2)) <= 0):\n        raise ValueError("Every actor must contribute videos")\n    if actor_equal_weight:\n        totals = a.sum((1, 2))[:, None, None]\n        a, b = a / totals, b / totals\n    rng = np.random.default_rng(seed)\n    n = len(a)\n    multiplicities = rng.multinomial(n, np.full(n, 1 / n), size=n_boot)\n    samples_a = metrics_from_confusion((multiplicities @ a.reshape(n, 36)).reshape(n_boot, 6, 6))\n    samples_b = metrics_from_confusion((multiplicities @ b.reshape(n, 36)).reshape(n_boot, 6, 6))\n    observed_a, observed_b = metrics_from_confusion(a.sum(0)), metrics_from_confusion(b.sum(0))\n    rows, distributions = [], {}\n    for metric in ("accuracy", "macro_f1"):\n        delta = samples_a[metric] - samples_b[metric]\n        for target, point, samples in (("EmoT", observed_a[metric], samples_a[metric]), ("TemporalCNN1D", observed_b[metric], samples_b[metric]), ("EmoT_minus_TemporalCNN1D", observed_a[metric] - observed_b[metric], delta)):\n            lo, hi = np.percentile(samples, [2.5, 97.5])\n            rows.append({"estimand": "equal_actor_weight" if actor_equal_weight else "video_weighted", "metric": metric, "target": target, "observed": float(point), "ci_low": float(lo), "ci_high": float(hi), "n_boot": n_boot, "seed": seed, "n_actors": n})\n        distributions[metric + "_delta"] = delta\n    return pd.DataFrame(rows), pd.DataFrame(distributions)\n\n\ndef permute_actor_categories(categories, blocks, rng, fixed_categories=()):\n    """Permute complete actor categories within OOF fold blocks."""\n    categories, blocks = np.asarray(categories), np.asarray(blocks)\n    if len(categories) != len(blocks):\n        raise ValueError("Categories and fold blocks have different lengths")\n    result = categories.copy()\n    movable = ~np.isin(categories, tuple(fixed_categories))\n    for block in np.unique(blocks):\n        index = np.flatnonzero((blocks == block) & movable)\n        result[index] = rng.permutation(categories[index])\n    return result\n\n\ndef holm_adjust(p_values):\n    p = np.asarray(p_values, float)\n    order = np.argsort(p)\n    ranked = np.minimum(1, np.maximum.accumulate(p[order] * np.arange(len(p), 0, -1)))\n    result = np.empty(len(p));result[order] = ranked\n    return result\n\n\ndef demographic_permutations(cm, actors, n_perm=5000, seed=20261107, min_videos=30, min_actors=2, hold_unknown_fixed=False):\n    """Four planned accuracy-gap tests, with Holm correction across attributes."""\n    totals = cm.sum((1, 2)).astype(float)\n    correct = np.diagonal(cm, axis1=1, axis2=2).sum(1).astype(float)\n    blocks = actors.outer_test_fold.to_numpy()\n    rows, support_rows = [], []\n    for j, attr in enumerate(ATTRIBUTES):\n        labels = actors[attr].fillna("Unknown").astype(str).to_numpy()\n        eligible = []\n        for group in sorted(np.unique(labels)):\n            mask = labels == group\n            na, nv = int(mask.sum()), int(totals[mask].sum())\n            ok = na >= min_actors and nv >= min_videos and group.casefold() not in {\'unknown\',\'nan\',\'none\',\'\'}\n            support_rows.append({"attribute": attr, "group": group, "n_actors": na, "n_videos": nv, "eligible_for_permutation": ok, "low_actor_support_flag": na < 10})\n            if ok:eligible.append(group)\n        if len(eligible) < 2:\n            raise ValueError(f"Fewer than two eligible groups for {attr}")\n        def gap(cats):\n            accuracies = [correct[cats == group].sum() / totals[cats == group].sum() for group in eligible]\n            return float(max(accuracies) - min(accuracies))\n        observed = gap(labels)\n        rng = np.random.default_rng(seed + 100 + j)\n        null = np.array([gap(permute_actor_categories(labels, blocks, rng, fixed_categories=(\'Unknown\',) if hold_unknown_fixed else ())) for _ in range(n_perm)])\n        p_value = (1 + np.count_nonzero(null >= observed - 1e-14)) / (n_perm + 1)\n        rows.append({"attribute": attr, "observed_accuracy_gap": observed, "p_actor_permutation": p_value, "n_perm": n_perm, "seed": seed + 100 + j, "permutation_blocks": "outer_test_fold", "unknown_category_fixed":hold_unknown_fixed, "eligible_groups": "|".join(eligible), "min_actors": min_actors, "min_videos": min_videos})\n    result = pd.DataFrame(rows)\n    result["p_holm_four_attributes"] = holm_adjust(result.p_actor_permutation)\n    return result, pd.DataFrame(support_rows)\n\n\ndef group_emotion_metrics(by_model):\n    group_rows, emotion_rows = [], []\n    for model, df in by_model.items():\n        for attr in ATTRIBUTES:\n            for group, g in df.groupby(attr, sort=True):\n                cm = confusion(g.y_true, g.y_pred)\n                met = metrics_from_confusion(cm)\n                prob = g[list(PROB_COLUMNS)].to_numpy(float)\n                confidence = prob.max(1)\n                correct = g.y_true.to_numpy() == g.y_pred.to_numpy()\n                bins = np.minimum((confidence * 10).astype(int), 9)\n                ece = 0.\n                for bi in range(10):\n                    mask = bins == bi\n                    if mask.any():ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())\n                group_rows.append({"model": model, "attribute": attr, "group": group, "n_videos": len(g), "n_actors": g.ActorID.nunique(), **{key:float(value) for key,value in met.items()}, "ece_10_bins": ece})\n                for k, emotion in enumerate(CLASSES):\n                    true = g.y_true.to_numpy() == k\n                    pred = g.y_pred.to_numpy() == k\n                    tp = int((true & pred).sum())\n                    n_true, n_pred = int(true.sum()), int(pred.sum())\n                    recall = tp / n_true if n_true else np.nan\n                    precision = tp / n_pred if n_pred else 0.\n                    emotion_rows.append({"model": model, "attribute": attr, "group": group, "emotion_id": k, "emotion": emotion, "n_videos_true": n_true, "n_actors_true": g.loc[true, "ActorID"].nunique(), "n_videos_predicted": n_pred, "true_positive": tp, "recall": recall, "precision": precision, "f1": 2*tp/(n_true+n_pred) if n_true+n_pred else 0.})\n    return pd.DataFrame(group_rows), pd.DataFrame(emotion_rows)\n\n\ndef gap_eligibility_sensitivity(group_metrics, min_videos=30):\n    """Preserve the historical rule and separately declare an actor-aware rule."""\n    rows=[]\n    for (model,attr),frame in group_metrics.groupby([\'model\',\'attribute\'],sort=True):\n        for policy in (\'historical_videos_ge30\',\'known_groups_actors_ge2_videos_ge30\'):\n            chosen=frame.loc[frame.n_videos>=min_videos]\n            if policy!=\'historical_videos_ge30\':\n                chosen=chosen.loc[(chosen.n_actors>=2)&~chosen.group.astype(str).str.casefold().isin([\'unknown\',\'nan\',\'none\',\'\'])]\n            if len(chosen)<2:raise ValueError(\'Fewer than two eligible groups\')\n            row={\'model\':model,\'attribute\':attr,\'eligibility_policy\':policy,\'groups\':\'|\'.join(sorted(chosen.group.astype(str))),\n                 \'n_groups\':len(chosen),\'minimum_group_actors\':int(chosen.n_actors.min()),\'minimum_group_videos\':int(chosen.n_videos.min())}\n            for metric in (\'accuracy\',\'macro_f1\'):\n                row[metric+\'_gap\']=float(chosen[metric].max()-chosen[metric].min())\n                row[\'worst_group_\'+metric]=float(chosen[metric].min())\n            rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef run_analysis(input_dir, output_dir, n_boot=5000, n_perm=5000, seed=20261107):\n    input_dir, output_dir = Path(input_dir), Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    names = ["cv_pooled_predictions.csv", "manifest_valid_with_splits.csv", "cv_actor_assignments.csv", "cv_fold_metrics.csv"]\n    inputs = {name:pd.read_csv(input_dir/name) for name in names}\n    by_model, actors = validate_inputs(inputs[names[0]], inputs[names[1]], inputs[names[2]], expected_samples=7441, expected_actors=91)\n    actor_ids = actors.ActorID.to_numpy()\n    matrices = {model:actor_confusions(df, actor_ids) for model, df in by_model.items()}\n    metrics_rows = []\n    for model, df in by_model.items():\n        for fold, part in df.groupby("fold"):\n            computed = metrics_from_confusion(confusion(part.y_true, part.y_pred))\n            saved = inputs["cv_fold_metrics.csv"].query("model == @model and fold == @fold")\n            if len(saved) != 1 or any(not np.isclose(float(computed[key]), float(saved.iloc[0][key]), atol=1e-12) for key in computed):\n                raise ValueError(f"Recomputed fold metrics disagree with record for {model} fold {fold}")\n            metrics_rows.append({"model":model, "fold":int(fold), "n_videos":len(part), "n_actors":part.ActorID.nunique(), **{key:float(value) for key,value in computed.items()}})\n    fold_metrics = pd.DataFrame(metrics_rows)\n    paired = fold_metrics.pivot(index="fold",columns="model",values="macro_f1")[["EmoT","TemporalCNN1D"]].reset_index()\n    paired["delta_macro_f1"] = paired.EmoT-paired.TemporalCNN1D\n    paired.to_csv(output_dir/\'paired_fold_macro_f1.csv\',index=False)\n    fold_metrics.to_csv(output_dir/\'fold_metrics_recomputed.csv\',index=False)\n    bootstrap_results = []\n    for equal in (False, True):\n        stats, samples = cluster_bootstrap_pair(matrices[\'EmoT\'],matrices[\'TemporalCNN1D\'],n_boot,seed,actor_equal_weight=equal)\n        bootstrap_results.append(stats)\n        samples.to_csv(output_dir/f"bootstrap_deltas_{\'equal_actor\' if equal else \'video_weighted\'}.csv",index=False)\n    boot = pd.concat(bootstrap_results,ignore_index=True)\n    boot.to_csv(output_dir/\'actor_cluster_bootstrap.csv\',index=False)\n    permutation, support = demographic_permutations(matrices[\'EmoT\'],actors,n_perm,seed,hold_unknown_fixed=True)\n    permutation.to_csv(output_dir/\'actor_demographic_permutation.csv\',index=False)\n    permutation_unknown_movable,_=demographic_permutations(matrices[\'EmoT\'],actors,n_perm,seed,hold_unknown_fixed=False)\n    permutation_unknown_movable.to_csv(output_dir/\'actor_demographic_permutation_unknown_movable_sensitivity.csv\',index=False)\n    support.to_csv(output_dir/\'actor_group_support.csv\',index=False)\n    group, emotion = group_emotion_metrics(by_model)\n    group.to_csv(output_dir/\'group_metrics_with_actor_support.csv\',index=False)\n    emotion.to_csv(output_dir/\'emotion_group_recall_with_actor_support.csv\',index=False)\n    gaps=gap_eligibility_sensitivity(group)\n    gaps.to_csv(output_dir/\'group_gap_eligibility_sensitivity.csv\',index=False)\n    gaps.query("attribute == \'Race\'").to_csv(output_dir/\'racial_gap_all_models_two_eligibility_rules.csv\',index=False)\n    mitigation=[]\n    for policy,frame in gaps.query("attribute == \'Race\'").groupby(\'eligibility_policy\'):\n        indexed=frame.set_index(\'model\')\n        base,weighted=indexed.loc[\'EmoT\'],indexed.loc[\'EmoT_Reweighted\']\n        mitigation.append({\'eligibility_policy\':policy,\'groups\':base[\'groups\'],\n            \'EmoT_accuracy_gap\':float(base.accuracy_gap),\'Reweighted_accuracy_gap\':float(weighted.accuracy_gap),\n            \'Reweighted_minus_EmoT_accuracy_gap\':float(weighted.accuracy_gap-base.accuracy_gap),\n            \'EmoT_macro_f1_gap\':float(base.macro_f1_gap),\'Reweighted_macro_f1_gap\':float(weighted.macro_f1_gap),\n            \'Reweighted_minus_EmoT_macro_f1_gap\':float(weighted.macro_f1_gap-base.macro_f1_gap)})\n    mitigation=pd.DataFrame(mitigation)\n    mitigation.to_csv(output_dir/\'mitigation_gap_sensitivity.csv\',index=False)\n    fold_support=[]\n    for attr in ATTRIBUTES:\n        for (fold,group_name),part in by_model[\'EmoT\'].groupby([\'fold\',attr],sort=True):\n            fold_support.append({\'fold\':int(fold),\'attribute\':attr,\'group\':group_name,\'n_actors\':int(part.ActorID.nunique()),\'n_videos\':len(part),\'low_actor_support_flag\':part.ActorID.nunique()<10})\n    pd.DataFrame(fold_support).to_csv(output_dir/\'actor_group_support_by_fold.csv\',index=False)\n    emotion_reference_audit=None\n    reference_path=input_dir/\'cv_group_emotion_metrics.csv\'\n    if reference_path.exists():\n        saved=pd.read_csv(reference_path).rename(columns={\'protected_attribute\':\'attribute\'})\n        keys=[\'model\',\'attribute\',\'group\',\'emotion_id\']\n        merged=emotion.merge(saved,on=keys,suffixes=(\'_recomputed\',\'_saved\'),validate=\'one_to_one\',how=\'outer\',indicator=True)\n        if not merged[\'_merge\'].eq(\'both\').all():raise ValueError(\'Emotion/group coverage differs from saved table\')\n        max_error=float(np.max(np.abs(merged.recall_recomputed-merged.recall_saved)))\n        if not np.allclose(merged.recall_recomputed,merged.recall_saved,atol=1e-12,equal_nan=True):raise ValueError(\'Saved group/emotion recalls were not reproduced\')\n        if not np.array_equal(merged.n_videos_true,merged.support_true):raise ValueError(\'Saved group/emotion supports were not reproduced\')\n        emotion_reference_audit={\'source\':reference_path.name,\'rows\':len(merged),\'max_abs_recall_error\':max_error,\'support_matches\':True,\'sha256\':hashlib.sha256(reference_path.read_bytes()).hexdigest()}\n    actor_metrics = []\n    for model, cm in matrices.items():\n        met = metrics_from_confusion(cm)\n        for i, actor in enumerate(actor_ids):\n            actor_metrics.append({"ActorID":int(actor),"model":model,"n_videos":int(cm[i].sum()),**{key:float(value[i]) for key,value in met.items()}})\n    pd.DataFrame(actor_metrics).to_csv(output_dir/\'metrics_per_actor.csv\',index=False)\n    summary = {\n      \'status\':\'completed\',\'analysis\':\'actor_cluster_reanalysis_of_saved_OOF_predictions\',\'run_id\':\'20260809_223452_c0613d3c\',\n      \'n_samples_per_model\':7441,\'n_actors\':91,\'models\':sorted(by_model),\'n_boot\':n_boot,\'n_perm\':n_perm,\'seed\':seed,\n      \'fold_audit\':\'Each sample once per model; same labels and folds as manifest and actor assignment; saved fold metrics independently reproduced.\',\n      \'estimand_primary\':\'Macro-F1/accuracy on pooled OOF videos; actor resampling keeps all videos of each sampled actor; identical multiplicities for the model pair.\',\n      \'estimand_sensitivity\':\'Pooled confusion matrix after normalizing each actor total to 1; each actor receives equal total weight. Not the average of actor-specific F1 values.\',\n      \'interval\':\'Percentile actor-cluster bootstrap, 2.5th and 97.5th percentiles; conditional on saved fitted models and actor sample; no new training.\',\n      \'demographic_test\':\'Accuracy gap among fixed eligible known groups (>=30 videos and >=2 actors); permute complete actor demographic categories within external fold; 4 p-values corrected by Holm. Main racial randomization fixes Unknown and its actor because Unknown denotes missing information, not a substantive racial category. A separately labelled sensitivity moves Unknown as a category in the null randomization. These are null randomizations, not demographic imputation. The actor remains included in other attributes and global metrics.\',\n      \'analysis_plan_status\':\'Escolhas metodológicas explicitadas nesta reanálise, posterior à dissertação original. Não há alegação de pré-registro.\',\n      \'limitations\':[\n       \'Os intervalos condicionais não incorporam a variabilidade de retreinamento, a dependência induzida pela sobreposição dos conjuntos de treino dos folds, deslocamentos para populações não observadas ou toda a incerteza da seleção de modelos.\',\n       \'A permutação demográfica pressupõe permutabilidade condicional das categorias dos atores dentro dos folds. A análise é observacional e associativa, sem identificação causal nem ajuste por outros atributos correlacionados.\',\n       \'A análise racial principal mantém Unknown e seu único ator fixos, fora das trocas, pois esse rótulo representa ausência de informação racial. O ator permanece nas outras comparações demográficas e nas métricas globais. Na sensibilidade, Unknown pode mudar de ator dentro do fold4 sob a hipótese nula; isso não altera nem imputa os metadados originais.\',\n       \'Grupos com poucos atores continuam muito incertos. O baixo suporte é sinalizado; o bootstrap não cria novas identidades.\',\n       \'Macro-F1 médio dos folds e macro-F1 das predições externas concatenadas são agregações diferentes. A diferença entre esses valores é esperada.\',\n       \'Não se afirma um valor p confirmatório a partir dos cinco folds, nem um valor p confirmatório de superioridade entre modelos.\'\n      ],\n      \'paired_fold_wins\':int((paired.delta_macro_f1>0).sum()),\'paired_fold_mean_delta\':float(paired.delta_macro_f1.mean()),\n      \'bootstrap\':boot.to_dict(orient=\'records\'),\'permutation\':permutation.to_dict(orient=\'records\'),\n      \'permutation_unknown_movable_sensitivity\':permutation_unknown_movable.to_dict(orient=\'records\'),\n      \'racial_gap_sensitivity\':gaps.query("attribute == \'Race\'").to_dict(orient=\'records\'),\n      \'mitigation_sensitivity\':mitigation.to_dict(orient=\'records\'),\n      \'saved_emotion_metrics_audit\':emotion_reference_audit,\n      \'inputs\':[{\'name\':name,\'sha256\':hashlib.sha256((input_dir/name).read_bytes()).hexdigest(),\'bytes\':(input_dir/name).stat().st_size} for name in names]\n    }\n    (output_dir/\'analysis_summary.json\').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    def num(v):return f\'{v:.6f}\'.replace(\'.\',\',\')\n    text = [\'# Reanálise por ator das predições externas preservadas\',\'\',\n      \'## Evidência e auditoria\',\'\',\n      \'Foram alinhadas 7.441 amostras por modelo com os 91 atores do manifesto. Cada vídeo aparece uma vez no teste externo de cada modelo. Rótulos, probabilidades, folds e metadados de ator foram conferidos. As métricas de cada fold foram recalculadas e coincidem com o CSV preservado. Não houve treinamento novo.\',\'\',\n      \'## Comparação pareada\',\'\',\n      f\'O EmoT vence a CNN1D nos {summary["paired_fold_wins"]} folds. A diferença média dos cinco macro-F1 é {num(summary["paired_fold_mean_delta"])}. Esse resultado descritivo não é apresentado como cinco experimentos independentes.\',\'\',\n      \'O estimando primário é o desempenho nas predições externas concatenadas, com peso dos vídeos. O bootstrap sorteia 91 atores com reposição e transporta todos os vídeos de cada ator. As multiplicidades de atores são idênticas para os dois modelos, em 5.000 réplicas, semente 20261107. Intervalos percentis de 95% são condicionais aos modelos externos já ajustados.\',\'\']\n    for _,row in boot.query("estimand == \'video_weighted\'").iterrows():\n        text.append(f\'- {row.target}, {row.metric}: {num(row.observed)}; IC condicional por ator [{num(row.ci_low)}; {num(row.ci_high)}].\')\n    text+=[\'\',\'## Sensibilidade ao peso por pessoa\',\'\',\n      \'A análise secundária normaliza a matriz de confusão de cada ator para massa total 1 antes de agregá-las. Cada pessoa recebe o mesmo peso; não se trata da média dos F1 individuais. Os dois estimandos respondem a perguntas diferentes.\',\'\']\n    for _,row in boot.query("estimand == \'equal_actor_weight\' and target == \'EmoT_minus_TemporalCNN1D\'").iterrows():\n        text.append(f\'- Diferença {row.metric}: {num(row.observed)}; IC condicional [{num(row.ci_low)}; {num(row.ci_high)}].\')\n    text+=[\'\',\'## Permutação demográfica por ator\',\'\',\n      \'As categorias completas dos atores foram permutadas dentro de cada fold externo, preservando as contagens de cada categoria por fold. A estatística é a amplitude de acurácia entre categorias conhecidas com pelo menos 30 vídeos e dois atores. As escolhas desta reanálise são posteriores à dissertação original, sem alegação de pré-registro. Foram usadas 5.000 permutações por atributo e p=(1+extremos)/(5001). O ajuste de Holm cobre a família de quatro atributos. Atributos correlacionados não são controlados causalmente.\',\n      \'Na análise racial principal, Unknown e seu único ator ficam fixos e fora das trocas: esse rótulo representa ausência de informação, não uma categoria racial substantiva. A decisão tem justificativa metodológica independente do valor p. O ator continua incluído nas análises dos outros atributos e nas métricas globais. A estatística racial usa somente os três grupos conhecidos elegíveis. A sensibilidade permite trocar também o rótulo Unknown dentro do fold4, gerando categorias hipotéticas sob uma hipótese nula mais ampla. Essa randomização não é imputação nem altera os metadados originais. Cada análise tem sua própria correção de Holm para os quatro atributos.\',\'\']\n    for _,row in permutation.iterrows():\n        text.append(f\'- {row.attribute}: gap={num(row.observed_accuracy_gap)}; p por ator={num(row.p_actor_permutation)}; p Holm={num(row.p_holm_four_attributes)}; grupos={row.eligible_groups}.\')\n    text+=[\'\',\'## Sensibilidade à regra de elegibilidade racial\',\'\',\n      \'A análise histórica admite grupos com pelo menos 30 vídeos. Em OOF isso inclui Unknown, que representa somente um ator e 82 vídeos. A regra alternativa é uma escolha metodológica explicitada nesta reanálise, posterior à dissertação: categorias conhecidas, pelo menos 30 vídeos e dois atores. Não há alegação de pré-registro. Essa regra produz outro estimando de disparidade, apresentado ao lado do histórico, sem substituição silenciosa. Dois atores é um mínimo operacional, não evidência de suporte estatístico suficiente. Asian tem somente sete atores no total; três folds contêm um ator Asian cada.\',\'\']\n    for _,row in gaps.query("attribute == \'Race\'").iterrows():\n        text.append(f\'- {row.eligibility_policy}, {row.model}: gap de acurácia={num(row.accuracy_gap)}; gap de macro-F1={num(row.macro_f1_gap)}; grupos={row.groups}.\')\n    text+=[\'\',\'Na comparação de mitigação, valores negativos abaixo significam redução da amplitude racial, não melhoria automática de todo o sistema.\',\'\']\n    for _,row in mitigation.iterrows():\n        text.append(f\'- {row.eligibility_policy}: Reweighted menos EmoT no gap de acurácia={num(row.Reweighted_minus_EmoT_accuracy_gap)}; no gap de macro-F1={num(row.Reweighted_minus_EmoT_macro_f1_gap)}.\')\n    text+=[\'\',\'Na sensibilidade que permite mover Unknown sob a hipótese nula, os resultados de permutação são:\',\'\']\n    for _,row in permutation_unknown_movable.iterrows():\n        text.append(f\'- {row.attribute}: p por ator={num(row.p_actor_permutation)}; p Holm desta família={num(row.p_holm_four_attributes)}.\')\n    text+=[\'\',\'## Suporte e resultados por emoção\',\'\',\n      \'Os CSVs incluem número de atores e vídeos por categoria e emoção. Grupos com menos de dez atores são sinalizados. A exclusão de uma categoria de um teste não elimina seus resultados descritivos. A categoria Unknown com suporte insuficiente em atores permanece nas tabelas descritivas.\',\'\',\n      \'## Limites da interpretação\',\'\']\n    text+=[\'- \'+limit for limit in summary[\'limitations\']]\n    text+=[\'\',\'A conclusão deve ser restrita ao estudo de caso e à análise condicional. Os intervalos de diferenças não substituem o pacote de sementes adicionais, ablação ou validação em outra base.\']\n    (output_dir/\'RELATORIO_ESTATISTICA_POR_ATOR.md\').write_text(\'\\n\'.join(text)+\'\\n\',encoding=\'utf-8\')\n    return summary\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--input-dir\',type=Path,required=True)\n    parser.add_argument(\'--output-dir\',type=Path,required=True)\n    parser.add_argument(\'--n-boot\',type=int,default=5000)\n    parser.add_argument(\'--n-perm\',type=int,default=5000)\n    parser.add_argument(\'--seed\',type=int,default=20261107)\n    args=parser.parse_args()\n    result=run_analysis(args.input_dir,args.output_dir,args.n_boot,args.n_perm,args.seed)\n    print(json.dumps({\'status\':result[\'status\'],\'n_actors\':result[\'n_actors\'],\'n_samples\':result[\'n_samples_per_model\'],\'mean_fold_delta\':result[\'paired_fold_mean_delta\'],\'output_dir\':str(args.output_dir)},ensure_ascii=False))\n', 'bruna_ravdess_canonical_extractor.py': '"""Canonical extraction functions from executed notebook08; no extraction runs on import.\nOnly the task-file helper is stricter: a preserved model is required, never downloaded as latest.\n"""\nfrom pathlib import Path\nimport numpy as np\nimport cv2\nimport mediapipe as mp\nfrom mediapipe.tasks import python as mp_tasks_python\nfrom mediapipe.tasks.python import vision as mp_tasks_vision\n\nMEDIAPIPE_BACKEND="tasks"\nmp_face_mesh=None\nBaseOptions=mp_tasks_python.BaseOptions\nFaceLandmarker=mp_tasks_vision.FaceLandmarker\nFaceLandmarkerOptions=mp_tasks_vision.FaceLandmarkerOptions\nVisionRunningMode=mp_tasks_vision.RunningMode\nT_FRAMES=100\nBASE_FEATURE_DIM=310\nMIN_FACE_DETECTION_RATE=0.60\nFACE_LANDMARKER_MODEL_PATH=None\n\ndef ensure_face_landmarker_model():\n    if FACE_LANDMARKER_MODEL_PATH is None or not Path(FACE_LANDMARKER_MODEL_PATH).is_file():\n        raise FileNotFoundError("Use the preserved CREMA-D face_landmarker.task; no latest model substitution.")\n    return Path(FACE_LANDMARKER_MODEL_PATH)\n\nSELECTED_LANDMARK_IDS=[0, 1, 2, 4, 5, 7, 10, 13, 14, 17, 21, 33, 37, 39, 40, 45, 46, 48, 52, 53, 54, 55, 58, 61, 63, 64, 65, 66, 67, 70, 78, 80, 81, 82, 84, 87, 88, 91, 93, 94, 95, 97, 98, 103, 105, 107, 109, 115, 127, 132, 133, 136, 144, 145, 146, 148, 149, 150, 152, 153, 154, 155, 157, 158, 159, 160, 161, 162, 163, 168, 172, 173, 176, 178, 181, 185, 191, 195, 197, 219, 220, 234, 235, 240, 246, 249, 251, 263, 267, 269, 270, 275, 276, 278, 282, 283, 284, 285, 288, 290, 291, 293, 295, 296, 297, 300, 305, 308, 310, 311, 312, 314, 317, 318, 321, 323, 324, 326, 327, 332, 334, 336, 338, 344, 356, 361, 362, 365, 373, 374, 375, 377, 378, 379, 380, 381, 382, 384, 385, 386, 387, 388, 389, 390, 397, 398, 400, 402, 405, 409, 415, 439, 440, 454, 466]\n\ndef create_face_landmarker():\n    # Cria o detector de landmarks compatível com a API disponível.\n    if MEDIAPIPE_BACKEND == "solutions":\n        return mp_face_mesh.FaceMesh(\n            static_image_mode=False,\n            max_num_faces=1,\n            refine_landmarks=False,\n            min_detection_confidence=0.5,\n            min_tracking_confidence=0.5,\n        )\n\n    model_path = ensure_face_landmarker_model()\n    options = FaceLandmarkerOptions(\n        base_options=BaseOptions(model_asset_path=str(model_path)),\n        running_mode=VisionRunningMode.IMAGE,\n        num_faces=1,\n        min_face_detection_confidence=0.5,\n        min_face_presence_confidence=0.5,\n        min_tracking_confidence=0.5,\n        output_face_blendshapes=False,\n        output_facial_transformation_matrixes=False,\n    )\n    return FaceLandmarker.create_from_options(options)\n\ndef sample_video_frames(video_path, t_frames=T_FRAMES):\n    cap = cv2.VideoCapture(str(video_path))\n    frames = []\n    while True:\n        ok, frame = cap.read()\n        if not ok:\n            break\n        frames.append(frame)\n    cap.release()\n    if not frames:\n        return []\n    idx = np.linspace(0, len(frames) - 1, t_frames).round().astype(int)\n    return [frames[i] for i in idx]\n\ndef normalize_landmarks_xy(coords_xy):\n    center = coords_xy.mean(axis=0, keepdims=True)\n    centered = coords_xy - center\n    span = coords_xy.max(axis=0) - coords_xy.min(axis=0)\n    return centered / max(float(span.max()), 1e-6)\n\ndef extract_frame_landmarks(frame_bgr, face_landmarker):\n    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)\n    if MEDIAPIPE_BACKEND == "solutions":\n        result = face_landmarker.process(rgb)\n        if not result.multi_face_landmarks:\n            return None\n        landmarks = result.multi_face_landmarks[0].landmark\n    else:\n        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb))\n        result = face_landmarker.detect(mp_image)\n        if not result.face_landmarks:\n            return None\n        landmarks = result.face_landmarks[0]\n    coords = np.array([[landmarks[idx].x, landmarks[idx].y] for idx in SELECTED_LANDMARK_IDS], dtype=np.float32)\n    return normalize_landmarks_xy(coords).reshape(-1).astype(np.float32)\n\ndef interpolate_missing_frames(seq):\n    valid = ~np.isnan(seq).any(axis=1)\n    if valid.sum() == 0:\n        return None\n    if valid.all():\n        return seq.astype(np.float32)\n    x = np.arange(seq.shape[0])\n    out = seq.copy()\n    for d in range(seq.shape[1]):\n        out[~valid, d] = np.interp(x[~valid], x[valid], seq[:, d][valid])\n    return out.astype(np.float32)\n\ndef process_one_video(row, face_landmarker):\n    frames = sample_video_frames(row["video_path"], T_FRAMES)\n    if not frames:\n        return None, {"file_stem": row["file_stem"], "status": "no_frames", "n_frames_read": 0, "detected_frames": 0, "detection_rate": 0.0}\n    seq = np.full((T_FRAMES, BASE_FEATURE_DIM), np.nan, dtype=np.float32)\n    detected = 0\n    for i, frame in enumerate(frames):\n        landmarks = extract_frame_landmarks(frame, face_landmarker)\n        if landmarks is not None:\n            seq[i] = landmarks\n            detected += 1\n    rate = detected / T_FRAMES\n    if rate < MIN_FACE_DETECTION_RATE:\n        return None, {"file_stem": row["file_stem"], "status": "low_detection_rate", "n_frames_read": len(frames), "detected_frames": detected, "detection_rate": rate}\n    seq = interpolate_missing_frames(seq)\n    if seq is None:\n        return None, {"file_stem": row["file_stem"], "status": "interpolation_failed", "n_frames_read": len(frames), "detected_frames": detected, "detection_rate": rate}\n    return seq, {"file_stem": row["file_stem"], "status": "ok", "n_frames_read": len(frames), "detected_frames": detected, "detection_rate": rate}\n', 'bruna_ravdess_external.py': '"""Frozen CREMA-D -> RAVDESS visual evaluation. Nothing executes on import."""\nfrom __future__ import annotations\n\nimport gc\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\nimport time\nfrom urllib.parse import urlparse\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\n\nfrom bruna_revision_experiments import (\n    CLASSES, CANONICAL_RUN, temporal_features, metrics, hash_file,\n    atomic_csv, atomic_json, atomic_npz, lock_config, fingerprint,\n)\nfrom bruna_revision_statistics import confusion, metrics_from_confusion\n\nPROTOCOL = \'bruna-ravdess-frozen-external-20261107-v2-env-recovery\'\nLABEL_MAP = {5: 0, 7: 1, 6: 2, 3: 3, 1: 4, 4: 5}\n\n\ndef parse_ravdess(filename):\n    stem = Path(filename).stem\n    match = re.fullmatch(r\'(\\d{2})-(\\d{2})-(\\d{2})-(\\d{2})-(\\d{2})-(\\d{2})-(\\d{2})\', stem)\n    if not match:\n        return None\n    modality, channel, emotion, intensity, statement, repetition, actor = map(int, match.groups())\n    if modality not in (1,2,3) or channel not in (1,2) or emotion not in range(1,9):\n        raise ValueError(\'Invalid RAVDESS modality/channel/emotion: \'+stem)\n    if actor not in range(1,25) or statement not in (1,2) or repetition not in (1,2) or intensity not in (1,2):\n        raise ValueError(\'Invalid RAVDESS actor or stimulus metadata: \'+stem)\n    if emotion == 1 and intensity != 1:\n        raise ValueError(\'Neutral has no strong intensity: \'+stem)\n    if modality != 2 or channel != 1 or emotion not in LABEL_MAP:\n        return None\n    return {\'dataset\': \'RAVDESS\', \'file_stem\': stem, \'ActorID\': actor,\n            \'identity_namespace\': f\'RAVDESS:{actor:02d}\', \'y\': LABEL_MAP[emotion],\n            \'label_name\': CLASSES[LABEL_MAP[emotion]], \'emotion_code\': emotion,\n            \'intensity\': intensity, \'statement\': statement, \'repetition\': repetition,\n            \'modality\': modality, \'vocal_channel\': channel}\n\n\ndef expected_stems():\n    return {f\'02-01-{e:02d}-{i:02d}-{s:02d}-{r:02d}-{a:02d}\'\n            for a in range(1,25) for e in LABEL_MAP\n            for i in ((1,) if e == 1 else (1,2)) for s in (1,2) for r in (1,2)}\n\n\ndef validate_manifest(manifest):\n    if \'file_stem\' not in manifest or manifest.file_stem.duplicated().any():\n        raise ValueError(\'Missing or duplicated external stimuli.\')\n    if set(manifest.file_stem) != expected_stems() or len(manifest) != 1056:\n        raise ValueError(\'The complete 24-actor/1,056-stimulus external set is required.\')\n    for row in manifest.to_dict(\'records\'):\n        parsed = parse_ravdess(row[\'file_stem\']+\'.mp4\')\n        if parsed is None or any(row.get(k) != parsed[k] for k in (\'ActorID\',\'y\',\'label_name\')):\n            raise ValueError(\'Manifest metadata differs from official filename.\')\n\n\ndef validate_checkpoint_normalization(payload):\n    if list(payload.get(\'class_names\', [])) != CLASSES:\n        raise ValueError(\'Checkpoint class order differs from CREMA-D.\')\n    mu, sigma = [np.asarray(payload[k], dtype=np.float32).reshape(-1) for k in (\'mu\',\'sigma\')]\n    if mu.shape != (1240,) or sigma.shape != (1240,) or not np.isfinite(mu).all() or not np.isfinite(sigma).all() or (sigma <= 0).any():\n        raise ValueError(\'Invalid saved CREMA-D training normalization.\')\n    return mu, sigma\n\n\ndef validate_external_alignment(manifest, x, cache, canonical_ids):\n    if manifest.file_stem.duplicated().any() or not len(manifest):\n        raise ValueError(\'Duplicated or empty external manifest.\')\n    for row in manifest.to_dict(\'records\'):\n        parsed=parse_ravdess(row[\'file_stem\']+\'.mp4\')\n        if parsed is None or any(row.get(k)!=parsed[k] for k in (\'ActorID\',\'identity_namespace\',\'y\',\'label_name\')):\n            raise ValueError(\'External metadata differs from stimulus identity.\')\n    if x.shape!=(len(manifest),100,310) or not np.isfinite(x).all():\n        raise ValueError(\'Invalid external input tensor.\')\n    if not np.array_equal(manifest.file_stem.to_numpy(str),cache[\'file_stems\'].astype(str)):\n        raise ValueError(\'External manifest/cache stimulus order mismatch.\')\n    if not np.array_equal(manifest.y.to_numpy(int),cache[\'y\']):\n        raise ValueError(\'External manifest/cache label mismatch.\')\n    if not np.array_equal(x,cache[\'X_base\']):\n        raise ValueError(\'Passed external tensor differs from the persisted cache.\')\n    if len(canonical_ids)!=155 or len(np.unique(canonical_ids))!=155 or not np.array_equal(cache[\'selected_landmark_ids\'],canonical_ids):\n        raise ValueError(\'External/canonical landmark IDs or order differ.\')\n\n\ndef verify_canonical_extraction(base_dir, output_dir, atol=1e-4, mean_atol=1e-6):\n    """Check six original clips before downloading an external dataset."""\n    import importlib.metadata\n    import bruna_ravdess_canonical_extractor as extractor\n    from bruna_revision_experiments import _torch_load\n    from bruna_revision_models import EmoTransformer, TemporalCNN1D\n    import torch\n    base,root=Path(base_dir),Path(output_dir)\n    root.mkdir(parents=True,exist_ok=True)\n    if importlib.metadata.version(\'mediapipe\')!=\'0.10.35\' or extractor.cv2.__version__!=\'4.13.0\':\n        raise RuntimeError(\'Restart after installing MediaPipe 0.10.35 and OpenCV headless 4.13.0.92.\')\n    task=base/\'cache/face_landmarker.task\'\n    if not task.is_file():raise FileNotFoundError(\'Preserved canonical task model missing.\')\n    extractor.FACE_LANDMARKER_MODEL_PATH=task\n    source=base/\'runs\'/CANONICAL_RUN/\'outputs\'\n    manifest=pd.read_csv(source/\'tables/manifest_valid_with_splits.csv\')\n    if len(manifest)!=7441 or manifest.file_stem.duplicated().any():\n        raise ValueError(\'Canonical manifest changed.\')\n    rng=np.random.default_rng(20261107);selected=[];used=set()\n    for label in range(6):\n        candidates=manifest.index[manifest.y.eq(label) & ~manifest.ActorID.isin(used)].to_numpy()\n        chosen=int(rng.choice(candidates));selected.append(chosen);used.add(manifest.loc[chosen,\'ActorID\'])\n    with np.load(base/\'cache/cremad_landmarks_T100_semantic.npz\',allow_pickle=False) as cache:\n        if not np.array_equal(cache[\'file_stems\'].astype(str),manifest.file_stem.to_numpy(str)) or not np.array_equal(cache[\'y\'],manifest.y.to_numpy(int)):\n            raise ValueError(\'Canonical tensor order/labels differ from its manifest.\')\n        if not np.array_equal(cache[\'selected_landmark_ids\'],extractor.SELECTED_LANDMARK_IDS):\n            raise ValueError(\'Canonical landmark selection differs from the extractor.\')\n        original=cache[\'X_base\'][selected].copy()\n    # Reject checkpoint/schema/runtime problems before any external download.\n    for name in (\'EmoT\',\'TemporalCNN1D\'):\n        payload=_torch_load(source/f\'models/best_{name}.pt\',torch)\n        validate_checkpoint_normalization(payload)\n        model=EmoTransformer(D=1240,n_classes=6) if name==\'EmoT\' else TemporalCNN1D(T=100,D=1240,n_classes=6)\n        model.load_state_dict(payload[\'model_state_dict\'],strict=True)\n        del model,payload\n    records=[]\n    with extractor.create_face_landmarker() as detector:\n        for reference,index in zip(original,selected):\n            row=manifest.loc[index].to_dict()\n            if not Path(row[\'video_path\']).is_file():raise FileNotFoundError(row[\'video_path\'])\n            sequence,report=extractor.process_one_video(row,detector)\n            if sequence is None:\n                records.append({\'file_stem\':row[\'file_stem\'],\'status\':report[\'status\'],\'match\':False})\n                continue\n            difference=np.abs(reference-sequence)\n            maximum,mean=float(difference.max()),float(difference.mean())\n            records.append({\'file_stem\':row[\'file_stem\'],\'ActorID\':int(row[\'ActorID\']),\'y\':int(row[\'y\']),\n                            \'status\':report[\'status\'],\'max_abs_difference\':maximum,\'mean_abs_difference\':mean,\n                            \'match\':maximum<=atol and mean<=mean_atol,\'video_sha256\':hash_file(row[\'video_path\'])})\n    frame=pd.DataFrame(records);atomic_csv(root/\'canonical_extractor_preflight.csv\',frame)\n    evidence={\'all_six_passed\':bool(frame.match.all()),\'selection_seed\':20261107,\'n_samples\':6,\n              \'max_atol\':atol,\'mean_atol\':mean_atol,\'mediapipe\':\'0.10.35\',\'task_sha256\':hash_file(task),\n              \'canonical_run\':CANONICAL_RUN,\'extractor_sha256\':hash_file(extractor.__file__),\'opencv\':extractor.cv2.__version__,\n              \'manifest_sha256\':hash_file(source/\'tables/manifest_valid_with_splits.csv\'),\n              \'samples\':records,\'scope\':\'numerical extraction consistency only; not a new performance estimate\'}\n    atomic_json(root/\'canonical_extractor_preflight.json\',evidence)\n    if not evidence[\'all_six_passed\']:\n        raise RuntimeError(\'Re-extraction differs from preserved CREMA-D tensors; investigate before external evaluation.\')\n    return evidence\n\n\ndef conditional_actor_bootstrap(y, predictions, actors, n_boot=5000, seed=20261107):\n    y, actors = np.asarray(y, dtype=int), np.asarray(actors)\n    if y.ndim != 1 or actors.shape != y.shape or not len(y) or not predictions or n_boot < 2:\n        raise ValueError(\'Invalid actor bootstrap input.\')\n    identities = np.unique(actors)\n    if len(identities) < 2:\n        raise ValueError(\'At least two external identities are required for an interval.\')\n    matrices = {}\n    for model, predicted in predictions.items():\n        predicted = np.asarray(predicted, dtype=int)\n        if predicted.shape != y.shape:\n            raise ValueError(\'External predictions are not paired.\')\n        matrices[model] = np.stack([confusion(y[actors==a], predicted[actors==a]) for a in identities])\n    rng = np.random.default_rng(seed)\n    # One shared multiplicity matrix preserves model pairing at actor level.\n    weights = rng.multinomial(len(identities), np.ones(len(identities))/len(identities), size=n_boot)\n    distribution, point = {}, {}\n    for model, cm in matrices.items():\n        distribution[model] = metrics_from_confusion(np.einsum(\'ba,aij->bij\',weights,cm))\n        point[model] = metrics_from_confusion(cm.sum(0))\n    records = []\n    names = list(predictions)\n    contrasts = [(m,None) for m in names] + [(names[i],names[j]) for i in range(len(names)) for j in range(i+1,len(names))]\n    for left, right in contrasts:\n        for metric in (\'accuracy\',\'macro_f1\'):\n            values = distribution[left][metric] - (distribution[right][metric] if right else 0)\n            estimate = point[left][metric] - (point[right][metric] if right else 0)\n            lo,hi = np.quantile(values,[.025,.975])\n            records.append({\'contrast\': f\'{left} minus {right}\' if right else left, \'metric\':metric,\n                            \'estimate\':float(estimate),\'ci_low\':float(lo),\'ci_high\':float(hi),\n                            \'n_actors\':len(identities),\'n_videos\':len(y),\'n_boot\':n_boot,\'seed\':seed,\n                            \'scope\':\'conditional on frozen fitted models and successfully extracted clips\'})\n    return pd.DataFrame(records)\n\n\ndef _download_archive(record, destination):\n    import requests\n    parsed = urlparse(record[\'url\'])\n    if parsed.scheme != \'https\' or parsed.hostname != \'zenodo.org\':\n        raise ValueError(\'Expected the official HTTPS Zenodo archive endpoint.\')\n    algorithm, expected = record[\'checksum\'].split(\':\',1)\n    if algorithm != \'md5\' or not re.fullmatch(\'[0-9a-f]{32}\',expected):\n        raise ValueError(\'Unexpected official archive checksum format.\')\n    if destination.is_file() and destination.stat().st_size == int(record[\'size\']):\n        digest=hashlib.md5()\n        with destination.open(\'rb\') as handle:\n            for chunk in iter(lambda:handle.read(2**20),b\'\'):digest.update(chunk)\n        if digest.hexdigest() == expected:\n            return\n    partial=destination.with_suffix(\'.download.partial\')\n    error=None\n    for attempt in range(3):\n        try:\n            digest=hashlib.md5(); size=0\n            with requests.get(record[\'url\'],stream=True,timeout=(30,180)) as response:\n                response.raise_for_status()\n                with partial.open(\'wb\') as handle:\n                    for chunk in response.iter_content(2**20):\n                        if chunk:handle.write(chunk);digest.update(chunk);size+=len(chunk)\n            if size != int(record[\'size\']) or digest.hexdigest() != expected:\n                raise ValueError(\'Official RAVDESS archive checksum/size mismatch.\')\n            partial.replace(destination)\n            return\n        except Exception as exc:\n            error=exc\n            print(f\'Archive attempt {attempt+1}/3 failed: {type(exc).__name__}\',flush=True)\n    raise RuntimeError(\'Unable to obtain verified RAVDESS archive.\') from error\n\n\ndef prepare_ravdess_videos(output_dir, archive_manifest, scratch_dir=\'/content/bruna_ravdess_archives\'):\n    root=Path(output_dir); raw=root/\'raw\'; raw.mkdir(parents=True,exist_ok=True)\n    scratch=Path(scratch_dir);scratch.mkdir(parents=True,exist_ok=True)\n    entries=archive_manifest[\'files\']\n    expected_names={f\'Video_Speech_Actor_{a:02d}.zip\' for a in range(1,25)}\n    if len(entries)!=24 or {r[\'name\'] for r in entries} != expected_names:\n        raise ValueError(\'Official manifest must contain exactly 24 speech archives.\')\n    config={\'version\':PROTOCOL,\'archive_manifest\':archive_manifest,\'selection\':\'VO02/speech01/six common classes; no calm-neutral merge\'}\n    fp=lock_config(raw,config)\n    rows=[]; excluded=[]\n    for entry in sorted(entries,key=lambda r:r[\'name\']):\n        actor=int(entry[\'name\'][-6:-4])\n        marker=raw/f\'actor_{actor:02d}.json\'\n        if marker.is_file():\n            existing=json.loads(marker.read_text())\n            if existing.get(\'fingerprint\')!=fp:raise ValueError(\'Incompatible raw-video resume.\')\n            for record in existing[\'videos\']:\n                path=Path(record[\'video_path\'])\n                if not path.is_file() or hash_file(path)!=record[\'video_sha256\']:\n                    raise ValueError(\'Persisted raw video changed; do not reuse this extraction.\')\n            rows.extend(existing[\'videos\']);excluded.extend(existing[\'excluded\'])\n            print(f\'RAVDESS actor {actor:02d}: verified resume\',flush=True)\n            continue\n        if shutil.disk_usage(scratch).free < int(entry[\'size\'])*2:\n            raise RuntimeError(\'Insufficient temporary disk for the next actor archive.\')\n        archive=scratch/entry[\'name\']\n        print(\'Downloading official archive:\',entry[\'name\'],flush=True)\n        _download_archive(entry,archive)\n        actor_rows=[];actor_excluded=[];seen=set()\n        with zipfile.ZipFile(archive) as bundle:\n            for item in bundle.infolist():\n                if item.is_dir() or not item.filename.lower().endswith(\'.mp4\'):continue\n                name=Path(item.filename).name\n                parsed=parse_ravdess(name)\n                if parsed is None:\n                    actor_excluded.append({\'archive\':entry[\'name\'],\'filename\':item.filename,\'reason\':\'outside predeclared modality/channel/classes\'})\n                    continue\n                if parsed[\'ActorID\'] != actor or parsed[\'file_stem\'] in seen:\n                    raise ValueError(\'Actor mismatch or duplicated stimulus inside archive.\')\n                seen.add(parsed[\'file_stem\'])\n                # Never use an archive-provided path as a destination.\n                target=raw/f\'Actor_{actor:02d}\'/(parsed[\'file_stem\']+\'.mp4\')\n                target.parent.mkdir(parents=True,exist_ok=True)\n                temporary=target.with_suffix(\'.mp4.partial\')\n                with bundle.open(item) as source, temporary.open(\'wb\') as dest:\n                    shutil.copyfileobj(source,dest,2**20)\n                temporary.replace(target)\n                parsed.update(video_path=str(target),video_sha256=hash_file(target),archive_name=entry[\'name\'])\n                actor_rows.append(parsed)\n        if seen != {s for s in expected_stems() if s.endswith(f\'-{actor:02d}\')}:\n            raise ValueError(\'Actor archive does not contain all 44 selected stimuli.\')\n        atomic_json(marker,{\'fingerprint\':fp,\'videos\':actor_rows,\'excluded\':actor_excluded})\n        rows.extend(actor_rows);excluded.extend(actor_excluded)\n        # Only this scratch archive is removed, after verified video persistence.\n        archive.unlink()\n        print(f\'RAVDESS actor {actor:02d}: {len(actor_rows)} eligible clips persisted\',flush=True)\n    frame=pd.DataFrame(rows).sort_values(\'file_stem\').reset_index(drop=True)\n    validate_manifest(frame)\n    atomic_csv(root/\'manifest_nominal.csv\',frame)\n    atomic_csv(root/\'excluded_stimuli.csv\',pd.DataFrame(excluded))\n    return frame\n\n\ndef extract_ravdess(base_dir, output_dir, manifest):\n    import importlib.metadata\n    from bruna_ravdess_canonical_extractor import process_one_video, create_face_landmarker\n    import bruna_ravdess_canonical_extractor as extractor\n    base,root=Path(base_dir),Path(output_dir)\n    validate_manifest(manifest)\n    version=importlib.metadata.version(\'mediapipe\')\n    if version!=\'0.10.35\' or extractor.cv2.__version__!=\'4.13.0\':\n        raise RuntimeError(\'Extraction requires the pinned MediaPipe 0.10.35/OpenCV 4.13.0 configuration and a passing preflight.\')\n    task=base/\'cache/face_landmarker.task\'\n    if not task.is_file():raise FileNotFoundError(\'The preserved canonical FaceLandmarker model is required; no latest-model download is substituted.\')\n    with np.load(base/\'cache/cremad_landmarks_T100_semantic.npz\',allow_pickle=False) as cache:\n        ids=cache[\'selected_landmark_ids\'].astype(int)\n    if not np.array_equal(ids,extractor.SELECTED_LANDMARK_IDS):\n        raise ValueError(\'The 155 selected landmark IDs/order differ from CREMA-D.\')\n    extractor.FACE_LANDMARKER_MODEL_PATH=task\n    config={\'protocol\':PROTOCOL,\'mediapipe\':version,\'opencv\':extractor.cv2.__version__,\'backend\':\'tasks/IMAGE/CPU\',\n            \'canonical_task_sha256\':hash_file(task),\'extractor_source_sha256\':hash_file(extractor.__file__),\n            \'manifest_records\':manifest[[\'file_stem\',\'video_sha256\']].to_dict(\'records\'),\n            \'T\':100,\'minimum_detection_rate\':.60,\'selected_landmark_ids\':ids.tolist()}\n    shards=root/\'extraction_shards\';fp=lock_config(shards,config)\n    reports=[];valid_stems=[];sequences=[]\n    with create_face_landmarker() as detector:\n        for index,row in enumerate(manifest.to_dict(\'records\')):\n            path=shards/(row[\'file_stem\']+\'.npz\');meta=path.with_suffix(\'.json\')\n            if meta.is_file():\n                saved=json.loads(meta.read_text())\n                if saved.get(\'fingerprint\')!=fp or hash_file(path)!=saved[\'sha256\']:\n                    raise ValueError(\'Invalid external-extraction resume.\')\n                report=saved[\'report\']\n                with np.load(path,allow_pickle=False) as z:\n                    sequence=z[\'sequence\']\n                if report[\'status\']==\'ok\' and (sequence.shape!=(100,310) or not np.isfinite(sequence).all()):\n                    raise ValueError(\'Invalid persisted external landmarks.\')\n            else:\n                if hash_file(row[\'video_path\']) != row[\'video_sha256\']:\n                    raise ValueError(\'Raw video changed after the frozen manifest.\')\n                sequence,report=process_one_video(row,detector)\n                # An unexpected extractor error aborts instead of becoming a silent exclusion.\n                if sequence is not None and (sequence.shape!=(100,310) or not np.isfinite(sequence).all()):\n                    raise ValueError(\'Unexpected landmark shape or nonfinite values.\')\n                atomic_npz(path,sequence=sequence if sequence is not None else np.empty((0,310),np.float32))\n                atomic_json(meta,{\'fingerprint\':fp,\'report\':report,\'sha256\':hash_file(path)})\n            reports.append({**row,**report})\n            if report[\'status\']==\'ok\':\n                sequences.append(sequence);valid_stems.append(row[\'file_stem\'])\n            if (index+1)%10==0:print(f\'RAVDESS extraction: {index+1}/1056; valid={len(valid_stems)}\',flush=True)\n    if not sequences:raise RuntimeError(\'No external clip met the canonical detection threshold.\')\n    report_frame=pd.DataFrame(reports)\n    atomic_csv(root/\'processing_report.csv\',report_frame)\n    valid=report_frame[report_frame.status.eq(\'ok\')].reset_index(drop=True)\n    if valid.file_stem.tolist()!=valid_stems:raise ValueError(\'External tensor/manifest alignment failed.\')\n    atomic_csv(root/\'manifest_valid.csv\',valid)\n    coverage=report_frame.groupby([\'ActorID\',\'label_name\']).agg(n_nominal=(\'status\',\'size\'),n_valid=(\'status\',lambda s:int(s.eq(\'ok\').sum()))).reset_index()\n    atomic_csv(root/\'coverage_actor_emotion.csv\',coverage)\n    x=np.stack(sequences).astype(np.float32)\n    atomic_npz(root/\'ravdess_landmarks_T100_semantic.npz\',X_base=x,y=valid.y.to_numpy(int),file_stems=np.asarray(valid_stems),selected_landmark_ids=ids)\n    atomic_json(root/\'extraction_completed.json\',{\'fingerprint\':fp,\'n_nominal\':1056,\'n_processed\':len(reports),\'n_valid\':len(valid),\'n_valid_actors\':int(valid.ActorID.nunique()),\'cache_sha256\':hash_file(root/\'ravdess_landmarks_T100_semantic.npz\'),\'valid_manifest_sha256\':hash_file(root/\'manifest_valid.csv\'),\'processing_report_sha256\':hash_file(root/\'processing_report.csv\')})\n    return valid,x\n\n\ndef evaluate_external(base_dir, output_dir, valid_manifest, x_base, n_boot=5000):\n    import torch\n    from bruna_revision_models import EmoTransformer, TemporalCNN1D\n    from bruna_revision_experiments import _torch_load, _predict, _environment\n    base,root=Path(base_dir),Path(output_dir)\n    with np.load(base/\'cache/cremad_landmarks_T100_semantic.npz\',allow_pickle=False) as canonical:\n        canonical_ids=canonical[\'selected_landmark_ids\']\n    with np.load(root/\'ravdess_landmarks_T100_semantic.npz\',allow_pickle=False) as cache:\n        validate_external_alignment(valid_manifest,x_base,cache,canonical_ids)\n    persisted=pd.read_csv(root/\'manifest_valid.csv\')\n    if not persisted[[\'file_stem\',\'ActorID\',\'y\']].equals(valid_manifest[[\'file_stem\',\'ActorID\',\'y\']].reset_index(drop=True)):\n        raise ValueError(\'Passed external manifest differs from persisted manifest.\')\n    report=pd.read_csv(root/\'processing_report.csv\');validate_manifest(report)\n    accepted=report.loc[report.status.eq(\'ok\'),\'file_stem\'].tolist()\n    if accepted!=valid_manifest.file_stem.tolist():raise ValueError(\'Valid subset differs from the full processing report.\')\n    extraction=json.loads((root/\'extraction_completed.json\').read_text())\n    for key,path in [(\'cache_sha256\',root/\'ravdess_landmarks_T100_semantic.npz\'),(\'valid_manifest_sha256\',root/\'manifest_valid.csv\'),(\'processing_report_sha256\',root/\'processing_report.csv\')]:\n        if extraction[key]!=hash_file(path):raise ValueError(\'External extraction artifact hash mismatch.\')\n    run=base/\'runs\'/CANONICAL_RUN/\'outputs\'\n    checkpoints={\'EmoT\':run/\'models/best_EmoT.pt\',\'TemporalCNN1D\':run/\'models/best_TemporalCNN1D.pt\'}\n    config={\'protocol\':PROTOCOL,\'training_dataset\':\'CREMA-D only\',\'target_dataset\':\'RAVDESS\',\n            \'canonical_run\':CANONICAL_RUN,\'checkpoint_sha256\':{k:hash_file(p) for k,p in checkpoints.items()},\n            \'cache_sha256\':hash_file(root/\'ravdess_landmarks_T100_semantic.npz\'),\n            \'valid_manifest_sha256\':hash_file(root/\'manifest_valid.csv\'),\'processing_report_sha256\':hash_file(root/\'processing_report.csv\'),\n            \'extraction_completion_sha256\':hash_file(root/\'extraction_completed.json\'),\n            \'source_sha256\':hash_file(__file__),\'model_source_sha256\':hash_file(Path(__file__).with_name(\'bruna_revision_models.py\')),\n            \'environment\':_environment(torch),\'normalization\':\'saved CREMA-D training mean/std; no target fitting\',\n            \'selection\':\'all predeclared successfully extracted clips; no RAVDESS tuning\',\'n_boot\':int(n_boot)}\n    results=root/\'evaluation\';fp=lock_config(results,config)\n    predictions={};summary=[];class_rows=[];actor_rows=[]\n    x=temporal_features(x_base);device=\'cuda\' if torch.cuda.is_available() else \'cpu\'\n    y=valid_manifest.y.to_numpy(int)\n    for name,path in checkpoints.items():\n        payload=_torch_load(path,torch);mu,sigma=validate_checkpoint_normalization(payload)\n        standardized=(x-mu)/sigma\n        if not np.isfinite(standardized).all():raise ValueError(\'Invalid fixed-normalization inputs.\')\n        model=(EmoTransformer(D=1240,n_classes=6) if name==\'EmoT\' else TemporalCNN1D(T=100,D=1240,n_classes=6)).to(device)\n        model.load_state_dict(payload[\'model_state_dict\'],strict=True)\n        pred,prob,logits=_predict(model,standardized,torch,device,batch_size=32)\n        predictions[name]=pred\n        frame=valid_manifest[[\'file_stem\',\'ActorID\',\'identity_namespace\',\'y\',\'label_name\']].copy()\n        frame[\'model\']=name;frame[\'y_pred\']=pred\n        for i,label in enumerate(CLASSES):frame[\'prob_\'+label]=prob[:,i]\n        atomic_csv(results/(name+\'_predictions.csv\'),frame)\n        atomic_npz(results/(name+\'_logits.npz\'),logits=logits,probabilities=prob,y_true=y,y_pred=pred,file_stems=valid_manifest.file_stem.to_numpy(str))\n        summary.append({\'model\':name,\'n_videos\':len(y),\'n_actors\':int(valid_manifest.ActorID.nunique()),**metrics(y,pred)})\n        cm=confusion(y,pred);atomic_csv(results/(name+\'_confusion.csv\'),pd.DataFrame(cm,index=CLASSES,columns=CLASSES).reset_index(names=\'true_class\'))\n        for i,label in enumerate(CLASSES):\n            support=int(cm[i].sum())\n            class_rows.append({\'model\':name,\'emotion\':label,\'n_videos\':support,\'n_actors\':int(valid_manifest.loc[valid_manifest.y.eq(i),\'ActorID\'].nunique()),\'true_positive\':int(cm[i,i]),\'recall\':float(cm[i,i]/support) if support else None})\n        for actor,part in frame.groupby(\'ActorID\'):\n            actor_rows.append({\'model\':name,\'ActorID\':int(actor),\'n_videos\':len(part),**metrics(part.y,part.y_pred)})\n        del model,payload,standardized;gc.collect()\n        if device==\'cuda\':torch.cuda.empty_cache()\n        print(\'External inference complete:\',name,summary[-1],flush=True)\n    atomic_csv(results/\'metrics.csv\',pd.DataFrame(summary))\n    atomic_csv(results/\'recall_by_emotion.csv\',pd.DataFrame(class_rows))\n    atomic_csv(results/\'metrics_per_actor.csv\',pd.DataFrame(actor_rows))\n    intervals=conditional_actor_bootstrap(y,predictions,valid_manifest.ActorID.to_numpy(),n_boot=n_boot)\n    atomic_csv(results/\'actor_cluster_intervals.csv\',intervals)\n    extraction=json.loads((root/\'extraction_completed.json\').read_text())\n    if extraction[\'n_nominal\']!=1056 or extraction[\'n_processed\']!=1056:\n        raise RuntimeError(\'Incomplete external extraction; completion cannot be declared.\')\n    artifact_records=[{\'path\':str(p.relative_to(results)),\'bytes\':p.stat().st_size,\'sha256\':hash_file(p)} for p in sorted(results.rglob(\'*\')) if p.is_file() and p.name!=\'completed_external_evaluation.json\']\n    completion={\'status\':\'completed_external_evaluation\',\'fingerprint\':fp,\'scope\':\'frozen original CREMA-D checkpoints; post-development cross-dataset evaluation\',\n                \'n_nominal\':1056,\'n_valid\':len(y),\'n_failed_detection\':1056-len(y),\'n_valid_actors\':int(valid_manifest.ActorID.nunique()),\n                \'gpu_used\':device==\'cuda\',\'no_finetuning\':True,\'n_boot\':n_boot,\'artifacts\':artifact_records,\n                \'limitations\':[\'RAVDESS was used earlier in project development\',\'Intervals conditional on frozen fits and valid detections\',\'No racial/ethnic/age attributes inferred\',\'Different from within-RAVDESS training/testing protocols\']}\n    atomic_json(results/\'completed_external_evaluation.json\',completion)\n    return completion\n'}
EXPECTED_HASHES={'bruna_revision_models.py': '8d238a5186aed96190fd4e84b392b5dba95973d49e444c9029550e610169ea8e', 'bruna_revision_experiments.py': 'ca8b0d4f37d515beebbbbeea999e35bc2bf36afd96599dba49d8070f7485ebba', 'bruna_revision_statistics.py': 'ef8107761cd49d80ffdeaf882d717adbc923fbabe8bcf38fdae38aca2dd3d567', 'bruna_ravdess_canonical_extractor.py': '1c3485b86daf8c968054be4a29e737b576c2cf74b072cf2bb60ccdaac98cf9af', 'bruna_ravdess_external.py': '58c83c498c5553bcdd63b4569a5216f40f0f2679d38d36ac7e6d5e5ffaf57662'}
ARCHIVE_MANIFEST={'record_id': '1188976', 'doi': '10.5281/zenodo.1188976', 'fetched_utc': '2026-11-07T19:54:04.5201880Z', 'files': [{'name': 'Video_Speech_Actor_06.zip', 'size': 568517859, 'checksum': 'md5:3c42877921cc08cfb5c841a0f2cb94a7', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_06.zip/content'}, {'name': 'Video_Speech_Actor_09.zip', 'size': 523489249, 'checksum': 'md5:775da349c50bf915a1bbcb37379bd092', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_09.zip/content'}, {'name': 'Video_Speech_Actor_05.zip', 'size': 562966425, 'checksum': 'md5:b00e17d61374ef6ba86fef35d50a20fb', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_05.zip/content'}, {'name': 'Video_Speech_Actor_04.zip', 'size': 546560009, 'checksum': 'md5:013eed832af1f7e97082aae398a00dbe', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_04.zip/content'}, {'name': 'Video_Speech_Actor_01.zip', 'size': 552968353, 'checksum': 'md5:3c8ececaf392b4a9b11b32271f4f6d01', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_01.zip/content'}, {'name': 'Video_Speech_Actor_08.zip', 'size': 562573079, 'checksum': 'md5:bed5f43d9c18e177e34d4bc1ed9b6d77', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_08.zip/content'}, {'name': 'Video_Speech_Actor_07.zip', 'size': 565652404, 'checksum': 'md5:bdd29bfe082f80361ec6b589845c283d', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_07.zip/content'}, {'name': 'Video_Speech_Actor_13.zip', 'size': 501877549, 'checksum': 'md5:c5f2fd4fd77941636947620368c612b5', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_13.zip/content'}, {'name': 'Video_Speech_Actor_12.zip', 'size': 557928718, 'checksum': 'md5:2a1f0ddc0ca207beedee6ce2ea863ad7', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_12.zip/content'}, {'name': 'Video_Speech_Actor_11.zip', 'size': 518634370, 'checksum': 'md5:824504aa41a8fd575e5459d5701dd378', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_11.zip/content'}, {'name': 'Video_Speech_Actor_03.zip', 'size': 567824665, 'checksum': 'md5:68fa240ddd8a3cc410c64efe5fb2b0e4', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_03.zip/content'}, {'name': 'Video_Speech_Actor_18.zip', 'size': 565655520, 'checksum': 'md5:fef2131ac361da182c0cb85f3d8e8a6c', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_18.zip/content'}, {'name': 'Video_Speech_Actor_17.zip', 'size': 551688845, 'checksum': 'md5:8b7ae3e5a85d2874ac8e1195e722e047', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_17.zip/content'}, {'name': 'Video_Speech_Actor_16.zip', 'size': 562416513, 'checksum': 'md5:a63c54570ebbe6bd616d0dced5630f7e', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_16.zip/content'}, {'name': 'Video_Speech_Actor_24.zip', 'size': 595799165, 'checksum': 'md5:eef90e806c9179bc5b4b098d03647ae3', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_24.zip/content'}, {'name': 'Video_Speech_Actor_23.zip', 'size': 545177865, 'checksum': 'md5:b31e8c8904e66102e7a951694db752d6', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_23.zip/content'}, {'name': 'Video_Speech_Actor_22.zip', 'size': 560907466, 'checksum': 'md5:fee55f72c9871401f23d4b167de3217d', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_22.zip/content'}, {'name': 'Video_Speech_Actor_15.zip', 'size': 525205576, 'checksum': 'md5:4b3b9bc8473e86f884630f2506684887', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_15.zip/content'}, {'name': 'Video_Speech_Actor_14.zip', 'size': 552484059, 'checksum': 'md5:466d31f4cce92a14b1f4ee884bbdf7d7', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_14.zip/content'}, {'name': 'Video_Speech_Actor_10.zip', 'size': 565580163, 'checksum': 'md5:70199d1c6902f76e17df308d5c41fa01', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_10.zip/content'}, {'name': 'Video_Speech_Actor_02.zip', 'size': 570673024, 'checksum': 'md5:a6f40d413b2e6ef25b3a595099e59abb', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_02.zip/content'}, {'name': 'Video_Speech_Actor_21.zip', 'size': 590292896, 'checksum': 'md5:d810cf2ff6863ef91306fc7a16d74fbb', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_21.zip/content'}, {'name': 'Video_Speech_Actor_20.zip', 'size': 561349264, 'checksum': 'md5:5ddfb22770093bafebdefe8faa449f48', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_20.zip/content'}, {'name': 'Video_Speech_Actor_19.zip', 'size': 581121687, 'checksum': 'md5:abb97d97d8c4e88c866cee1cea9d06d6', 'url': 'https://zenodo.org/api/records/1188976/files/Video_Speech_Actor_19.zip/content'}]}
CODE_DIR=Path('/content/bruna_ravdess_revision_code')
CODE_DIR.mkdir(exist_ok=True)
for name,source in MODULES.items():
    if hashlib.sha256(source.encode()).hexdigest()!=EXPECTED_HASHES[name]:raise RuntimeError('Código alterado: '+name)
    compile(source,name,'exec')
    (CODE_DIR/name).write_text(source,encoding='utf-8')
sys.path.insert(0,str(CODE_DIR))
from bruna_ravdess_external import verify_canonical_extraction, prepare_ravdess_videos, extract_ravdess, evaluate_external
from bruna_revision_experiments import atomic_json
atomic_json(OUTPUT_DIR/'notebook_code_manifest.json',{'canonical_notebook_sha256':'3e90e89858655dfbba39a1824ddf48c0b44004f82d0c33121d453fe15e224318','module_sha256':EXPECTED_HASHES})
atomic_json(OUTPUT_DIR/'official_archive_manifest.json',ARCHIVE_MANIFEST)
print('Módulos íntegros; manifesto oficial com',len(ARCHIVE_MANIFEST['files']),'arquivos.')


## 1. Conferir o extrator em vídeos originais CREMA-D

Antes do download externo, reextrair seis vídeos CREMA-D: uma classe por vídeo, com atores distintos e escolha determinística. Comparar com o tensor original. O notebook reutiliza a cópia preservada de `face_landmarker.task`, sem baixar um modelo chamado latest. Se a representação não coincidir dentro das tolerâncias registradas, a execução para e preserva o diagnóstico. Não afrouxar a tolerância para obter um resultado externo.


In [ ]:
preflight=verify_canonical_extraction(BASE_DIR,OUTPUT_DIR)
display(pd.read_csv(OUTPUT_DIR/'canonical_extractor_preflight.csv'))
assert preflight['all_six_passed']


## 2. Obter e validar os vídeos oficiais

Cada ZIP é verificado com tamanho e MD5 publicados pelo Zenodo. A seleção é auditada contra os 1.056 nomes esperados. O manifesto preserva hashes dos vídeos, identidade, classe, intensidade, frase e repetição. Não usa CSVs antigos de 68 marcos nem duplica versões com e sem áudio.


In [ ]:
nominal=prepare_ravdess_videos(OUTPUT_DIR,ARCHIVE_MANIFEST)
display(nominal.groupby(['ActorID','label_name']).size().unstack(fill_value=0))
assert len(nominal)==1056 and nominal.ActorID.nunique()==24


## 3. Extrair os mesmos 155 marcos

São as mesmas funções canônicas de amostragem, centralização/escala por quadro, interpolação e limiar de detecção de 60%, com 100 instantes. As falhas são registradas por vídeo e não desaparecem do denominador nominal. Uma falha inesperada interrompe a rotina em vez de virar exclusão silenciosa. A retomada confere configuração e hashes dos resultados por vídeo.


In [ ]:
valid,x_base=extract_ravdess(BASE_DIR,OUTPUT_DIR,nominal)
display(pd.read_csv(OUTPUT_DIR/'coverage_actor_emotion.csv'))
print('Vídeos nominais:',len(nominal),'válidos:',len(valid),'atores válidos:',valid.ActorID.nunique())


## 4. Inferência congelada e incerteza por ator

Aplicar posição e diferenças exatamente como no CREMA-D; padronizar somente com média e desvio-padrão do treino salvos em cada checkpoint. Exportar predições, métricas, confusão e revocação. Bootstrap de 5.000 amostras de atores, com as mesmas multiplicidades para os dois modelos. Os intervalos são condicionais aos ajustes congelados e aos vídeos cuja extração foi válida; não representam a variação de treinamento nem eliminam a limitação de cobertura.


In [ ]:
external=evaluate_external(BASE_DIR,OUTPUT_DIR,valid,x_base,n_boot=5000)
display(pd.read_csv(OUTPUT_DIR/'evaluation/metrics.csv'))
display(pd.read_csv(OUTPUT_DIR/'evaluation/actor_cluster_intervals.csv'))
display(pd.read_csv(OUTPUT_DIR/'evaluation/recall_by_emotion.csv'))
print(json.dumps(external,indent=2))


## 5. Figuras derivadas das predições externas reais

As figuras abaixo usam somente as tabelas recém-produzidas. Não substituem as figuras históricas do CREMA-D e não equiparam o teste entre bases ao protocolo de estudos treinados dentro do RAVDESS.


In [ ]:
import matplotlib.pyplot as plt
from bruna_revision_experiments import CLASSES, hash_file
FIG_DIR=OUTPUT_DIR/'figures'
FIG_DIR.mkdir(exist_ok=True)
labels=['Raiva','Nojo','Medo','Alegria','Neutro','Tristeza']
for model in ('EmoT','TemporalCNN1D'):
    cm=pd.read_csv(OUTPUT_DIR/f'evaluation/{model}_confusion.csv').iloc[:,1:].to_numpy(float)
    rates=np.divide(cm,cm.sum(1,keepdims=True),out=np.zeros_like(cm),where=cm.sum(1,keepdims=True)>0)
    fig,ax=plt.subplots(figsize=(7.4,6.4),layout='constrained')
    im=ax.imshow(rates,vmin=0,vmax=1,cmap='Blues')
    ax.set_xticks(range(6),labels,rotation=35,ha='right');ax.set_yticks(range(6),labels)
    ax.set_xlabel('Classe prevista');ax.set_ylabel('Classe verdadeira')
    ax.set_title(f'{model}: CREMA-D → RAVDESS\nTeste externo visual, sem ajuste no RAVDESS')
    for i in range(6):
        for j in range(6):ax.text(j,i,f'{int(cm[i,j])}\n{rates[i,j]:.2f}',ha='center',va='center',color='white' if rates[i,j]>.55 else 'black',fontsize=9)
    fig.colorbar(im,ax=ax,label='Fração dentro da classe verdadeira')
    fig.savefig(FIG_DIR/f'{model}_confusao_externa.pdf');fig.savefig(FIG_DIR/f'{model}_confusao_externa.png',dpi=220)
    plt.show();plt.close(fig)
recall=pd.read_csv(OUTPUT_DIR/'evaluation/recall_by_emotion.csv')
fig,ax=plt.subplots(figsize=(8,4.8),layout='constrained')
for offset,model in zip((-.18,.18),('EmoT','TemporalCNN1D')):
    values=recall[recall.model.eq(model)].set_index('emotion').loc[CLASSES,'recall'].to_numpy()
    ax.bar(np.arange(6)+offset,values,width=.36,label=model)
ax.set_xticks(range(6),labels);ax.set_ylim(0,1);ax.set_ylabel('Revocação')
ax.set_title('CREMA-D → RAVDESS: revocação por emoção');ax.legend(loc='upper left',bbox_to_anchor=(1.01,1))
fig.savefig(FIG_DIR/'revocacao_externa_por_emocao.pdf');fig.savefig(FIG_DIR/'revocacao_externa_por_emocao.png',dpi=220)
plt.show();plt.close(fig)
fig_manifest=[{'file':p.name,'sha256':hash_file(p),'bytes':p.stat().st_size} for p in sorted(FIG_DIR.iterdir()) if p.is_file()]
atomic_json(OUTPUT_DIR/'figure_manifest.json',fig_manifest)
print('Avaliação externa concluída; vídeos e cache permanecem separados do repositório de notebooks.')
